In [ ]:
import pandas as pd
import os
dataset_13847333_age_sql = """
    SELECT
        * 
    FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_ehr_data = 0 """

df_age = pd.read_gbq(
    dataset_13847333_age_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")
df_age.head(5)

In [ ]:
type_ = 'visit_occurrence'#only some have something-but no matching concept id
type__ = 'visit'
type___ = '_source'
type_ = 'condition_occurrence'
type__ = 'condition'
type___ = '_source'
type_ = 'measurement'#has something
type__ = type_
type___ = '_source'
type_ = 'drug_exposure'
type__ = 'drug'
type___ = '_type'
type_ = 'procedure_occurrence'#has something -- survey
type__ = 'procedure'
type___ = '_type'
type_ = 'observation'#has something --
type__ = type_
type___ = '_source'
type_ = 'device_exposure'
type__ = 'device'
type___ = '_type'
type_ = 'death'#has info
type__ = 'death_type'
type___ = ''
type_ = 'specimen'
type__ = 'specimen_type'
type___ = ''
dataset_condition_sql = f"""
SELECT
    *
FROM `{os.environ["WORKSPACE_CDR"]}.{type_}` c
LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` cc
    ON c.{type__}_concept_id = cc.concept_id
LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` cs
    ON c.{type__}{type___}_concept_id = cs.concept_id
WHERE c.person_id IN UNNEST({list(set(df_age['person_id']))[200000:]})
"""

# type_ = 'survey_conduct'#has something --
# dataset_condition_sql = f"""
# SELECT
#     *
# FROM `{os.environ["WORKSPACE_CDR"]}.{type_}` c
# WHERE c.person_id IN UNNEST({list(set(df_age['person_id']))[200:400]})
# """

both_df_all = pd.read_gbq(
    dataset_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
)
both_df_all.head(5)

In [ ]:
type_ = 'measurement'#has something for few (bmi, other things)
type__ = type_
type___ = '_source'
type_ = 'procedure_occurrence'#has something -mapping to a survey conducted for very very few
type__ = 'procedure'
type___ = '_type'
type_ = 'observation'#has for all mostly, smoking, employement
type__ = type_
type___ = '_source'
type_ = 'death'#has for all  deathinfo
type__ = 'death_type'
type___ = ''
dataset_condition_sql = f"""
SELECT
    *
FROM `{os.environ["WORKSPACE_CDR"]}.{type_}` c
LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` cc
    ON c.{type__}_concept_id = cc.concept_id
LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` cs
    ON c.{type__}{type___}_concept_id = cs.concept_id
WHERE c.person_id IN UNNEST({list(set(df_age['person_id']))[0:1000]})
"""

type_ = 'survey_conduct'#has for all mostly
dataset_condition_sql = f"""
SELECT
    *
FROM `{os.environ["WORKSPACE_CDR"]}.{type_}` c
WHERE c.person_id IN UNNEST({list(set(df_age['person_id']))[200:400]})
"""

both_df_all = pd.read_gbq(
    dataset_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
)
both_df_all.head(5)

In [ ]:
# len(set(both_df_all['person_id'])),len(both_df_all)

In [ ]:
# both_df_all['concept_name_1'].value_counts()
# both_df_all['survey_source_value'].value_counts()

In [ ]:
# c= 0
# for i,j in zip([0,100000,200000],[100000,200000,239951]):
#     type_ = 'survey_conduct'#has for all mostly
#     dataset_condition_sql = f"""
#     SELECT
#         *
#     FROM `{os.environ["WORKSPACE_CDR"]}.{type_}` c
#     WHERE c.person_id IN UNNEST({list(set(df_age['person_id']))[i:j]})
#     """

#     both_df_all = pd.read_gbq(
#         dataset_condition_sql,
#         dialect="standard",
#         use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
#     )
#     c+=len(set(both_df_all['person_id']))
# c


cj= 0
for i,j in zip([0,100000,200000],[100000,200000,239951]):
    type_ = 'death'
    type__ = 'death_type'
    type___ = ''
    type_ = 'observation'#has for all mostly, smoking, employement
    type__ = type_
    type___ = '_source'
    type_ = 'procedure_occurrence'#has something -mapping to a survey conducted for very very few
    type__ = 'procedure'
    type___ = '_type'
    type_ = 'measurement'#has something for few (bmi, other things)
    type__ = type_
    type___ = '_source'
    type_ = 'visit_occurrence'#only some have something-but no matching concept id
    type__ = 'visit'
    type___ = '_source'
    dataset_condition_sql = f"""
    SELECT
        *
    FROM `{os.environ["WORKSPACE_CDR"]}.{type_}` c
    LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` cc
        ON c.{type__}_concept_id = cc.concept_id
    LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` cs
        ON c.{type__}{type___}_concept_id = cs.concept_id
    WHERE c.person_id IN UNNEST({list(set(df_age['person_id']))[i:j]})
    """
    both_df_all = pd.read_gbq(
        dataset_condition_sql,
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    )
    cj+=len(set(both_df_all['person_id']))
cj


### BCS 2.0

In [ ]:
codes_t = {}
dfs_survey = {}
# codes_t['alcohol'] = [1586201]
# codes_t['smoking'] = [1585860,1585870]
codes_t['alcohol'] = {'q1':1586201}
codes_t['smoking'] = {'q1':1585860,'q2':1585870}
codes_t['stress'] = {'q1':1332854}
codes_t['social'] = {'q1':1585735}
codes_t['meaning_in_life'] = {'q1':1332750}
codes_t['hearing'] = {'q1':836825}
codes_t['vig_pa'] = {'q1':1333286,'q2':1332870,'q3':903631}
codes_t['mod_pa'] = {'q1':1333288,'q2':1332871,'q3':903629}
codes_t['sleep'] = {'q1':1333275,'q2':1332752,'q3':836816}
codes_t['bp'] = {'q1':836787,'q2':43530310,'q3':1384437}
codes_t['glucose'] = {'q1':836800,'q2':43528820,'q3':836799,'q4':43528819,'q5':836848,'q6':43528816}
codes_t['cholesterol'] = {'q1':836836,'q2':43528793}

In [ ]:
import pandas as pd
import os
import numpy as np


# sur_type = 'alcohol'
# sur_type = 'smoking'
for sur_type in ['alcohol','smoking','stress','social','meaning_in_life','hearing',\
                'vig_pa','mod_pa','sleep','bp','glucose','cholesterol']:
    dataset_84558060_survey_sql = f"""
        SELECT
            answer.person_id,
            answer.survey_datetime,
            answer.survey,
            answer.question_concept_id,
            answer.question,
            answer.answer_concept_id,
            answer.answer,
            answer.survey_version_concept_id,
            answer.survey_version_name  
        FROM
            `{os.environ["WORKSPACE_CDR"]}.ds_survey` answer   
        WHERE
            (
                question_concept_id IN UNNEST({list(codes_t[sur_type].values())})
            )"""

    dfs_survey[sur_type] = pd.read_gbq(
        dataset_84558060_survey_sql,
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook")

dfs_survey[sur_type].head(5)

In [ ]:
dataset_13847333_age_sql = """
    SELECT
        * 
    FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            """

df_age_f = pd.read_gbq(
    dataset_13847333_age_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")
df_age_f.head(5)
df_age_f['age_date'] = (
    pd.to_datetime(df_age_f['dob']) +
    pd.to_timedelta(df_age_f['age_at_consent'] * 365.25, unit='D')
)
df_age_f['age_date_2'] = (
    pd.to_datetime(df_age_f['dob']) + pd.to_timedelta(2 * 365.24, unit='D') +
    pd.to_timedelta(df_age_f['age_at_consent'] * 365.25, unit='D')
)
df_age_f['age_date_1'] = (
    pd.to_datetime(df_age_f['dob']) + pd.to_timedelta(1 * 365.24, unit='D') +
    pd.to_timedelta(df_age_f['age_at_consent'] * 365.25, unit='D')
)

In [ ]:
#do not run
for kk in ['alcohol','smoking','stress','social','meaning_in_life',
 'hearing','vig_pa','mod_pa','sleep','bp','glucose','cholesterol']:
    temp_df = dfs_survey[kk].copy()
    temp_df = temp_df.merge(df_age_f[['person_id','age_date','age_date_1','age_date_2']],on = 'person_id')
    temp_df['survey_datetime'] = temp_df['survey_datetime'].apply(lambda x: str(x).split(' ')[0])
    temp_df['age_date'] = temp_df['age_date'].apply(lambda x: str(x).split(' ')[0])
    temp_df['age_date_1'] = temp_df['age_date_1'].apply(lambda x: str(x).split(' ')[0])
    temp_df['age_date_2'] = temp_df['age_date_2'].apply(lambda x: str(x).split(' ')[0])
    temp_df['survey_datetime'] = pd.to_datetime(temp_df['survey_datetime'])
    temp_df['age_date'] = pd.to_datetime(temp_df['age_date'])
    temp_df['age_date_1'] = pd.to_datetime(temp_df['age_date_1'])
    temp_df['age_date_2'] = pd.to_datetime(temp_df['age_date_2'])
    print(kk,len(temp_df[temp_df['survey_datetime']<=temp_df['age_date']]),\
    len(temp_df[temp_df['survey_datetime']<=temp_df['age_date_2']]),\
    len(temp_df[temp_df['survey_datetime']<=temp_df['age_date_1']]),\
    len(temp_df),\
    len(temp_df[temp_df['survey_datetime']<=temp_df['age_date']].groupby('person_id')\
        .agg({'survey_datetime':'max'}).reset_index()),\
    len(temp_df[temp_df['survey_datetime']<=temp_df['age_date_1']].groupby('person_id')\
        .agg({'survey_datetime':'max'}).reset_index()),\
    len(temp_df[temp_df['survey_datetime']<=temp_df['age_date_2']].groupby('person_id')\
        .agg({'survey_datetime':'max'}).reset_index()),\
    len(temp_df.groupby('person_id')\
        .agg({'survey_datetime':'max'}).reset_index()))

In [ ]:
#do not run
import pandas as pd
import matplotlib.pyplot as plt

data = [
    ["alcohol", 1390, 513580, 518352, 1390, 502256, 513580, 518352],
    ["smoking", 1185, 418072, 421693, 630, 220550, 224878, 226757],
    ["stress", 28, 78308, 119711, 20, 25481, 50773, 70689],
    ["social", 1551, 577962, 582671, 1551, 566645, 577962, 582671],
    ["meaning_in_life", 28, 78308, 119711, 20, 25481, 50773, 70689],
    ["hearing", 11811, 153219, 168049, 11738, 96478, 128925, 140759],
    ["vig_pa", 54, 144724, 222087, 20, 25481, 50773, 70689],
    ["mod_pa", 62, 172852, 265697, 20, 25481, 50773, 70689],
    ["sleep", 12439, 266374, 329108, 12291, 124005, 159741, 175196],
    ["bp", 6585, 547043, 614970, 5700, 156665, 204782, 228000],
    ["glucose", 41849, 466716, 513408, 16342, 147314, 178799, 195144],
    ["cholesterol", 7711, 402679, 448234, 7075, 141922, 186719, 206690],
]

df = pd.DataFrame(data, columns=[
    "metric",
    "count_age",
    "count_age2",
    "total_rows",
    "unique_age",
    "unique_age1",
    "unique_age2",
    "unique_total"
])

plt.figure()
plt.plot(df["metric"], df["unique_age"], marker='o', label="survey date ≤ start_fu")
plt.plot(df["metric"], df["unique_age1"], marker='o', label="survey date ≤ start_fu + 1y")
plt.plot(df["metric"], df["unique_age2"], marker='o', label="survey date ≤ start_fu + 2y")
plt.plot(df["metric"], df["unique_total"], marker='o', label="any survey date")

plt.xticks(rotation=45)
plt.xlabel("Metric")
plt.ylabel("Unique People")
plt.title("Data Availability by Time Window\n age_at_consent (start_fu)")
plt.legend()
plt.grid()

plt.show()


In [ ]:
for kk in ['alcohol','smoking','stress','social','meaning_in_life',
 'hearing','vig_pa','mod_pa','sleep','bp','glucose','cholesterol']:
    temp_df = dfs_survey[kk].copy()
    temp_df = temp_df.merge(df_age_f[['person_id','age_date','age_date_1','age_date_2']],on = 'person_id')
    temp_df['survey_datetime1'] = temp_df['survey_datetime'].apply(lambda x: str(x).split(' ')[0])
    temp_df['age_date'] = temp_df['age_date'].apply(lambda x: str(x).split(' ')[0])
#     temp_df['age_date_1'] = temp_df['age_date_1'].apply(lambda x: str(x).split(' ')[0])
    temp_df['age_date_2'] = temp_df['age_date_2'].apply(lambda x: str(x).split(' ')[0])
    temp_df['survey_datetime1'] = pd.to_datetime(temp_df['survey_datetime1'])
    temp_df['age_date'] = pd.to_datetime(temp_df['age_date'])
#     temp_df['age_date_1'] = pd.to_datetime(temp_df['age_date_1'])
    temp_df['age_date_2'] = pd.to_datetime(temp_df['age_date_2'])
    dfs_survey[f'{kk}_filtered'] = temp_df[temp_df['survey_datetime1']<=temp_df['age_date_2']]

In [ ]:
def latest_valid_answer(rows,INVALID_ANSWERS):
    if rows.empty:
        return None
    ans = rows.sort_values("survey_datetime").iloc[-1]["answer"]
    return None if ans in INVALID_ANSWERS else ans


In [ ]:
def alcohol_score(person_id, survey_df):
    rows = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"].isin([codes_t['alcohol']['q1']]))
    ]
    
    answer = latest_valid_answer(rows,['PMI: Prefer Not To Answer','PMI: Skip'])
    
    if answer is None:
        return 'exclude'
    elif '4 or More Per Week' in answer:
        return 0
    else:
        return 1
    
pid_q = 2765479
pid_q = 1449307    
alcohol_score(pid_q, dfs_survey['alcohol'])

In [ ]:
def smoking_score(person_id, survey_df):

    rows1 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"] == codes_t['smoking']['q1'])
    ]

    rows2 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"] == codes_t['smoking']['q2'])
    ]

    ans1 = latest_valid_answer(rows1,['PMI: Prefer Not To Answer','PMI: Skip','PMI: Dont Know'])
    ans2 = latest_valid_answer(rows2,['PMI: Prefer Not To Answer','PMI: Skip','PMI: Dont Know'])

    
    if ans1 is not None:
        # Q1 answered
        if 'Not at all' in ans1:
            return 5, 'not needed', 5
        else:
            return 0, 'not needed', 0

    # Q1 missing → fall back to Q2
    if ans2 is not None:
        return 'exclude', 5, 5

    # Nothing usable
    return 'exclude', 'exclude', 'exclude'
pid_q = 2765479
pid_q = 1449307  
pid_q = 1153605 
# pid_q = 2616415
# pid_q = 1857421  
smoking_score(pid_q, dfs_survey['smoking'])

In [ ]:

def stress_score(person_id, survey_df):
    rows = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"].isin([codes_t['stress']['q1']]))
    ]
    
    answer = latest_valid_answer(rows,['PMI: Skip'])
    if answer is None:
        return 'exclude'
    elif 'never' in answer.lower():
        return 3
    elif 'sometimes' in answer.lower():
        return 1
    else:
        return 0
    
pid_q = 2765479
pid_q = 1449307  
pid_q = 1153605
stress_score(pid_q, dfs_survey['stress'])

In [ ]:

def social_engagement_score(person_id, survey_df):
    rows = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"].isin([codes_t['social']['q1']]))
    ]

    answer = latest_valid_answer(rows,['PMI: Skip'])
    
    if answer is None:
        return 'exclude'
    
    elif 'good' in answer.lower() or 'excellent' in answer.lower():
        return 3
    else:
        return 0
pid_q = 2765479
pid_q = 1449307  
pid_q = 1153605 
pid_q = 2616415
social_engagement_score(pid_q, dfs_survey['social'])

In [ ]:

def meaning_in_life_score(person_id, survey_df):
    rows = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"].isin([codes_t['meaning_in_life']['q1']]))
    ]

    answer = latest_valid_answer(rows,['PMI: Skip','Prefer not to answer'])
    
    if answer is None:
        return 'exclude'
    
    elif 'moderate' in answer.lower() or 'extreme' in answer.lower() or 'very' in answer.lower():
        return 3
    else:
        return 0

pid_q = 2765479
pid_q = 1449307  
pid_q = 1153605 
pid_q = 2616415
pid_q = 1857421
meaning_in_life_score(pid_q, dfs_survey['meaning_in_life'])

In [ ]:


def hearing_impairment_score(person_id, survey_df):
    rows = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"].isin([codes_t['hearing']['q1']]))
    ]

    answer = latest_valid_answer(rows,['PMI: Skip','Prefer not to answer'])
    
    if answer is None:
        return 'exclude'
    
    elif 'self' in answer.lower():
        return 0
    else:
        return 3

    
pid_q = 2765479
pid_q = 1449307  
pid_q = 1153605 
pid_q = 2616415
pid_q = 1857421
hearing_impairment_score(pid_q, dfs_survey['hearing'])

In [ ]:


def physical_activity_vigorous_score(person_id, survey_df):

    rows1 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"] == codes_t['vig_pa']['q1'])
    ]
    rows2 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"] == codes_t['vig_pa']['q2'])
    ]
    rows3 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"] == codes_t['vig_pa']['q3'])
    ]

    ans1 = latest_valid_answer(rows1,['PMI: Skip'])
    ans2 = latest_valid_answer(rows2,['PMI: Skip'])
    ans3 = latest_valid_answer(rows3,['PMI: Skip'])

    # ---- Q1 ----
    if ans1 is None:
        return 'exclude', 'not needed', 'not needed', 'not needed', 'exclude'

    if 'yes' not in ans1.lower():
        return 0, 'not needed', 'not needed', 'not needed', 0

    # ---- Q2 ----
    if ans2 is None:
        return 'go to q2', 'exclude', 'not needed', 'not needed', 'exclude'

    days = int(ans2)

    # ---- Q3 ----
    if ans3 is None:
        return 'go to q2', days, 'exclude', 'not needed', 0

    minutes = int(ans3)
    amul = days * minutes

    final_answer = 3 if amul >= 75 else 0

    return 'go to q2', days, minutes, amul, final_answer


pid_q = 2765479
pid_q = 1449307  
pid_q = 1153605 
pid_q = 2616415
pid_q = 1857421
physical_activity_vigorous_score(pid_q, dfs_survey['vig_pa'])

In [ ]:

def physical_activity_moderate_score(person_id, survey_df):
    rows1 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"].isin([codes_t['mod_pa']['q1']]))
    ]
    
    rows2 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"].isin([codes_t['mod_pa']['q2']]))
    ]
    
    rows3 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"].isin([codes_t['mod_pa']['q3']]))
    ]

    ans1 = latest_valid_answer(rows1,['PMI: Skip'])
    ans2 = latest_valid_answer(rows2,['PMI: Skip'])
    ans3 = latest_valid_answer(rows3,['PMI: Skip'])

    # ---- Q1 ----
    if ans1 is None:
        return 'exclude', 'not needed', 'not needed', 'not needed', 'exclude'

    if 'yes' not in ans1.lower():
        return 0, 'not needed', 'not needed', 'not needed', 0

    # ---- Q2 ----
    if ans2 is None:
        return 'go to q2', 'exclude', 'not needed', 'not needed', 'exclude'

    days = int(ans2)

    # ---- Q3 ----
    if ans3 is None:
        return 'go to q2', days, 'exclude', 'not needed', 0

    minutes = int(ans3)
    amul = days * minutes

    final_answer = 3 if amul >= 150 else 0

    return 'go to q2', days, minutes, amul, final_answer
    
pid_q = 2765479
pid_q = 1449307  
pid_q = 1153605 
pid_q = 2616415
pid_q = 1857421
physical_activity_moderate_score(pid_q, dfs_survey['mod_pa'])

In [ ]:
def sleep_score(person_id, survey_df):

    rows1 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"] == codes_t['sleep']['q1'])
    ]
    rows2 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"] == codes_t['sleep']['q2'])
    ]
    rows3 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"] == codes_t['sleep']['q3'])
    ]

    ans1 = latest_valid_answer(rows1,['PMI: Skip'])
    ans2 = latest_valid_answer(rows2,['PMI: Skip'])
    ans3 = latest_valid_answer(rows3,['PMI: Skip'])

    # ---- a1 ----
    if ans1 is None:
        a1 = 'exclude'
    elif 'not' in ans1.lower():
        a1 = 3
    else:
        a1 = 0

    # ---- a2 ----
    a2 = 0 if ans2 is not None else 'exclude'

    # ---- a3 ----
    a3 = 0 if ans3 is not None else 'exclude'

    # ---- final ----
    vals = [a1, a2, a3]
    non_exclude = [v for v in vals if v != 'exclude']

    if not non_exclude:
        final_answer = 'exclude'
    elif a1 != 'exclude':
        final_answer = a1
    elif 3 in non_exclude:
        final_answer = 3
    else:
        final_answer = 0

    return a1, a2, a3, final_answer

pid_q = 2765479
pid_q = 1449307  
pid_q = 1153605 
pid_q = 2616415
pid_q = 1857421
sleep_score(pid_q, dfs_survey['sleep'])

In [ ]:
def latest_valid_row(rows, invalid_vals):
    if rows.empty:
        return None

    row = rows.sort_values("survey_datetime").iloc[-1]
    return None if row["answer"] in invalid_vals else row

def closest_value_in_window(valuedf, person_id, ref_date, days=30):
    """
    Select the measurement closest to ref_date within ±days.
    """
    df = valuedf[
        (valuedf["person_id"] == person_id) &
        (valuedf["measurement_datetime"].between(
            ref_date - pd.Timedelta(days=days),
            ref_date + pd.Timedelta(days=days)
        ))
    ]

    if df.empty:
        return None

    # Find closest date
    idx = (df["measurement_datetime"] - ref_date).abs().idxmin()
    row = df.loc[idx]

    return row


In [ ]:
import pandas
import os

# This query represents dataset "hba1c" for domain "measurement" and was generated for All of Us Controlled Tier Dataset v8
dataset_79363214_measurement_sql = """
    SELECT
        measurement.person_id,
        measurement.measurement_concept_id,
        m_standard_concept.concept_name as standard_concept_name,
        m_standard_concept.concept_code as standard_concept_code,
        m_standard_concept.vocabulary_id as standard_vocabulary,
        measurement.measurement_datetime,
        measurement.measurement_type_concept_id,
        m_type.concept_name as measurement_type_concept_name,
        measurement.operator_concept_id,
        m_operator.concept_name as operator_concept_name,
        measurement.value_as_number,
        measurement.value_as_concept_id,
        m_value.concept_name as value_as_concept_name,
        measurement.unit_concept_id,
        m_unit.concept_name as unit_concept_name,
        measurement.range_low,
        measurement.range_high,
        measurement.visit_occurrence_id,
        m_visit.concept_name as visit_occurrence_concept_name,
        measurement.measurement_source_value,
        measurement.measurement_source_concept_id,
        m_source_concept.concept_name as source_concept_name,
        m_source_concept.concept_code as source_concept_code,
        m_source_concept.vocabulary_id as source_vocabulary,
        measurement.unit_source_value,
        measurement.value_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.measurement` measurement 
        WHERE
            (
                measurement_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (3003309, 3004410, 3005673, 3007263, 3034639, 36032094, 36304734, 40762352, 4184637, 4197971, 42869630)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )) measurement 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_standard_concept 
            ON measurement.measurement_concept_id = m_standard_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_type 
            ON measurement.measurement_type_concept_id = m_type.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_operator 
            ON measurement.operator_concept_id = m_operator.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_value 
            ON measurement.value_as_concept_id = m_value.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_unit 
            ON measurement.unit_concept_id = m_unit.concept_id 
    LEFT JOIn
        `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
            ON measurement.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_visit 
            ON v.visit_concept_id = m_visit.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_source_concept 
            ON measurement.measurement_source_concept_id = m_source_concept.concept_id"""

hba1c_df_e = pandas.read_gbq(
    dataset_79363214_measurement_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")


hba1c_df_e = hba1c_df_e[['person_id','standard_concept_name','value_as_number',
       'value_as_concept_id', 'value_as_concept_name','measurement_concept_id','measurement_datetime']]
hba1c_df_e = hba1c_df_e.rename(columns = {'value_as_number':'hba1c'})
hba1c_df_e['measurement_datetime'] = hba1c_df_e['measurement_datetime'].apply(lambda x: str(x).split('+')[0])
hba1c_df_e['measurement_datetime'] = hba1c_df_e['measurement_datetime'].apply(lambda x: str(x).split(' ')[0])
hba1c_df_e['measurement_datetime'] = pd.to_datetime(hba1c_df_e['measurement_datetime'],format='mixed')
hba1c_df_e = hba1c_df_e.reset_index(drop=True)

In [ ]:
hba1c_df_e['standard_concept_name'].value_counts()

In [ ]:
import gc
gc.collect()

In [ ]:
def glucose_score_old(person_id, survey_df, qq1, qq2, hba1c):

    rows1 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"] == codes_t['glucose'][qq1])
    ]
    rows2 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"] == codes_t['glucose'][qq2])
    ]

    ans1 = latest_valid_answer(rows1,['PMI: Skip'])
    ans2 = latest_valid_answer(rows2,['PMI: Skip'])

    # ---- Q1 ----
    if ans1 is None:
        return 'exclude', 'not needed', 'exclude'

    # Self diabetes
    if 'self' in ans1.lower():
        # ---- Q2 ----
        if ans2 is None:
            return 'go to q2', 'exclude','exclude'
        if ans2 is not None and 'no' in ans2.lower():
            return 'go to q2', 0, 0
        # ---- HbA1c fallback ----
        if pd.notna(hba1c):
            final = 4 if hba1c < 6.5 else 1
            return 'go to q2', final, final

        # No way to resolve
        return 'go to q2', 'no close value to match', 'no close value to match'

    # ---- Not self ----
    return 4, 'not needed', 4


def glucose_score(person_id, survey_df, qq1, qq2, hba1c_df, window_days):

    rows1 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"] == codes_t['glucose'][qq1])
    ]
    rows2 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"] == codes_t['glucose'][qq2])
    ]

    ans1 = latest_valid_answer(rows1,['PMI: Skip'])
    

    # ---- Q1 ----
    if ans1 is None:
        return 'exclude', 'not needed', 'exclude'

    # Self cholesterol
    if 'self' in ans1.lower():
        # ---- Q2 ----
        
        def score_followup_hba1c_df(rows, hba1cdf, person_id, window_days):
            ans_row = latest_valid_row(rows, ['PMI: Skip'])
            if ans_row is None:
                return 'exclude','not needed'

            ans = ans_row["answer"]
            q_date = ans_row["survey_datetime"]

            q_date = pd.to_datetime(str(q_date).split(' ')[0])

            val_u = closest_value_in_window(
                hba1cdf,
                person_id,
                q_date,
                days=window_days
            )
            if val_u is not None:
                hb__a1c = val_u['hba1c']
            else:
                hb__a1c = None

            return ans, hb__a1c
        
        ans2,hba1c = score_followup_hba1c_df(rows2, hba1c_df, person_id, window_days=window_days)
        
        
        if ans2 == 'exclude':
            return 'go to q2', 'exclude','exclude'
        
        if ans2!= 'exclude' and 'no' in ans2.lower():
            return 'go to q2', 0, 0

        # ---- Lab fallback ----
        if pd.notna(hba1c):
            final = 4 if hba1c < 6.5 else 1
            return 'go to q2', final, final
        
        
        # No way to resolve
        return 'go to q2', 'no close value to match', 'no close value to match'

    # ---- Not self ----
    return 2, 'not needed', 2



pid_q = 2765479
# pid_q = 1449307  
# pid_q = 1153605 
# pid_q = 2616415
# pid_q = 1857421
pid_q = 1277645
# pid_q = 8157991
glucose_score_old(pid_q, dfs_survey['glucose'],'q1','q2',6.6),\
glucose_score_old(pid_q, dfs_survey['glucose'],'q3','q4',6.6),\
glucose_score_old(pid_q, dfs_survey['glucose'],'q5','q6',6.6),\
glucose_score_old(pid_q, dfs_survey['glucose'],'q1','q2',None),\
glucose_score_old(pid_q, dfs_survey['glucose'],'q1','q2',np.nan),\
glucose_score(pid_q, dfs_survey['glucose'],'q1','q2',hba1c_df_e,365.24),\
glucose_score(pid_q, dfs_survey['glucose'],'q3','q4',hba1c_df_e,365.24),\
glucose_score(pid_q, dfs_survey['glucose'],'q5','q6',hba1c_df_e,365.24)

In [ ]:
import pandas
import os

# This query represents dataset "cls" for domain "measurement" and was generated for All of Us Controlled Tier Dataset v8
dataset_63125669_measurement_sql = """
    SELECT
        measurement.person_id,
        measurement.measurement_concept_id,
        m_standard_concept.concept_name as standard_concept_name,
        m_standard_concept.concept_code as standard_concept_code,
        m_standard_concept.vocabulary_id as standard_vocabulary,
        measurement.measurement_datetime,
        measurement.measurement_type_concept_id,
        m_type.concept_name as measurement_type_concept_name,
        measurement.operator_concept_id,
        m_operator.concept_name as operator_concept_name,
        measurement.value_as_number,
        measurement.value_as_concept_id,
        m_value.concept_name as value_as_concept_name,
        measurement.unit_concept_id,
        m_unit.concept_name as unit_concept_name,
        measurement.range_low,
        measurement.range_high,
        measurement.visit_occurrence_id,
        m_visit.concept_name as visit_occurrence_concept_name,
        measurement.measurement_source_value,
        measurement.measurement_source_concept_id,
        m_source_concept.concept_name as source_concept_name,
        m_source_concept.concept_code as source_concept_code,
        m_source_concept.vocabulary_id as source_vocabulary,
        measurement.unit_source_value,
        measurement.value_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.measurement` measurement 
        WHERE
            (
                measurement_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (4008265, 40772590, 4299360)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )) measurement 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_standard_concept 
            ON measurement.measurement_concept_id = m_standard_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_type 
            ON measurement.measurement_type_concept_id = m_type.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_operator 
            ON measurement.operator_concept_id = m_operator.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_value 
            ON measurement.value_as_concept_id = m_value.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_unit 
            ON measurement.unit_concept_id = m_unit.concept_id 
    LEFT JOIn
        `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
            ON measurement.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_visit 
            ON v.visit_concept_id = m_visit.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_source_concept 
            ON measurement.measurement_source_concept_id = m_source_concept.concept_id"""

cls_df_e = pandas.read_gbq(
    dataset_63125669_measurement_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

cholesterol_df_e = cls_df_e[cls_df_e['standard_concept_name'].isin(['Cholesterol [Mass/volume] in Serum or Plasma',\
'Total cholesterol measurement','Cholesterol measurement','Serum total cholesterol measurement',\
 'Cholesterol [Presence] in Blood by Test strip'])][['person_id','standard_concept_name','value_as_number',
       'value_as_concept_id', 'value_as_concept_name','measurement_concept_id','measurement_datetime']]
cholesterol_df_e = cholesterol_df_e.rename(columns = {'value_as_number':'cholesterol'})
cholesterol_df_e['measurement_datetime'] = cholesterol_df_e['measurement_datetime'].apply(lambda x: str(x).split('+')[0])
cholesterol_df_e['measurement_datetime'] = cholesterol_df_e['measurement_datetime'].apply(lambda x: str(x).split(' ')[0])
cholesterol_df_e['measurement_datetime'] = pd.to_datetime(cholesterol_df_e['measurement_datetime'],format='mixed')
cholesterol_df_e = cholesterol_df_e.reset_index(drop=True)

In [ ]:
cholesterol_df_e['standard_concept_name'].value_counts()

In [ ]:
[i for i in list(cls_df_e['standard_concept_name'].value_counts().keys()) if \
 ('LDL' not in i) and ('HDL' not in i) and ('IDL' not in i)]

In [ ]:
del cls_df_e
import gc
gc.collect()

In [ ]:



def cholesterol_score_old(person_id, survey_df, cl):

    rows1 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"] == codes_t['cholesterol']['q1'])
    ]
    rows2 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"] == codes_t['cholesterol']['q2'])
    ]

    ans1 = latest_valid_answer(rows1,['PMI: Skip'])
    ans2 = latest_valid_answer(rows2,['PMI: Skip'])

    # ---- Q1 ----
    if ans1 is None:
        return 'exclude', 'not needed', 'exclude'

    # Self cholesterol
    if 'self' in ans1.lower():
        # ---- Q2 ----
        
        if ans2 is None:
            return 'go to q2', 'exclude','exclude'
        
        if ans2 is not None and 'no' in ans2.lower():
            return 'go to q2', 0, 0

        # ---- Lab fallback ----
        if pd.notna(cl):
            final = 1 if cl >= 200 else 2
            return 'go to q2', final, final
        
        
        # No way to resolve
        return 'go to q2', 'no close value to match', 'no close value to match'

    # ---- Not self ----
    return 2, 'not needed', 2

def cholesterol_score(person_id, survey_df, cl_df, window_days):

    rows1 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"] == codes_t['cholesterol']['q1'])
    ]
    rows2 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"] == codes_t['cholesterol']['q2'])
    ]

    ans1 = latest_valid_answer(rows1,['PMI: Skip'])
    

    # ---- Q1 ----
    if ans1 is None:
        return 'exclude', 'not needed', 'exclude'

    # Self cholesterol
    if 'self' in ans1.lower():
        # ---- Q2 ----
        
        def score_followup_cl_df(rows, cldf, person_id, window_days):
            ans_row = latest_valid_row(rows, ['PMI: Skip'])
            if ans_row is None:
                return 'exclude','not needed'

            ans = ans_row["answer"]
            q_date = ans_row["survey_datetime"]

            q_date = pd.to_datetime(str(q_date).split(' ')[0])

            val_u = closest_value_in_window(
                cldf,
                person_id,
                q_date,
                days=window_days
            )
            if val_u is not None:
                cls = val_u['cholesterol']
            else:
                cls = None

            return ans, cls
        
        ans2,cl = score_followup_cl_df(rows2, cl_df, person_id, window_days=window_days)
        
        
        if ans2 == 'exclude':
            return 'go to q2', 'exclude','exclude'
        
        if ans2!= 'exclude' and 'no' in ans2.lower():
            return 'go to q2', 0, 0

        # ---- Lab fallback ----
        if pd.notna(cl):
            final = 1 if cl >= 200 else 2
            return 'go to q2', final, final
        
        
        # No way to resolve
        return 'go to q2', 'no close value to match', 'no close value to match'

    # ---- Not self ----
    return 2, 'not needed', 2


pid_q = 2765479
# pid_q = 1449307  
# pid_q = 1153605 
# pid_q = 2616415
# pid_q = 1857421
# pid_q = 1277645
pid_q = 2063987
cholesterol_score_old(pid_q, dfs_survey['cholesterol'],190),\
cholesterol_score_old(pid_q, dfs_survey['cholesterol'],None),\
cholesterol_score_old(pid_q, dfs_survey['cholesterol'],np.nan),\
cholesterol_score_old(pid_q, dfs_survey['cholesterol'],220),\
cholesterol_score(pid_q, dfs_survey['cholesterol'],cholesterol_df_e,365.24)


In [ ]:
import pandas
import os

# This query represents dataset "bpd" for domain "measurement" and was generated for All of Us Controlled Tier Dataset v8
dataset_03430134_measurement_sql = """
    SELECT
        measurement.person_id,
        measurement.measurement_concept_id,
        m_standard_concept.concept_name as standard_concept_name,
        m_standard_concept.concept_code as standard_concept_code,
        m_standard_concept.vocabulary_id as standard_vocabulary,
        measurement.measurement_datetime,
        measurement.measurement_type_concept_id,
        m_type.concept_name as measurement_type_concept_name,
        measurement.operator_concept_id,
        m_operator.concept_name as operator_concept_name,
        measurement.value_as_number,
        measurement.value_as_concept_id,
        m_value.concept_name as value_as_concept_name,
        measurement.unit_concept_id,
        m_unit.concept_name as unit_concept_name,
        measurement.range_low,
        measurement.range_high,
        measurement.visit_occurrence_id,
        m_visit.concept_name as visit_occurrence_concept_name,
        measurement.measurement_source_value,
        measurement.measurement_source_concept_id,
        m_source_concept.concept_name as source_concept_name,
        m_source_concept.concept_code as source_concept_code,
        m_source_concept.vocabulary_id as source_vocabulary,
        measurement.unit_source_value,
        measurement.value_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.measurement` measurement 
        WHERE
            (
                measurement_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (3004249, 3012888, 4152194, 4154790)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )) measurement 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_standard_concept 
            ON measurement.measurement_concept_id = m_standard_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_type 
            ON measurement.measurement_type_concept_id = m_type.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_operator 
            ON measurement.operator_concept_id = m_operator.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_value 
            ON measurement.value_as_concept_id = m_value.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_unit 
            ON measurement.unit_concept_id = m_unit.concept_id 
    LEFT JOIn
        `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
            ON measurement.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_visit 
            ON v.visit_concept_id = m_visit.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_source_concept 
            ON measurement.measurement_source_concept_id = m_source_concept.concept_id"""

bloodp_df = pandas.read_gbq(
    dataset_03430134_measurement_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

bloodp_df.head(5)

In [ ]:
bloodp_df['standard_concept_name'].value_counts()

In [ ]:
sbp_df = bloodp_df[
    bloodp_df['standard_concept_name'].str.contains('systolic', case=False, na=False)
][['person_id','standard_concept_name','value_as_number',
       'value_as_concept_id', 'value_as_concept_name','measurement_concept_id','measurement_datetime']]
dbp_df = bloodp_df[
    bloodp_df['standard_concept_name'].str.contains('diastolic', case=False, na=False)
][['person_id','standard_concept_name','value_as_number',
       'value_as_concept_id', 'value_as_concept_name','measurement_concept_id','measurement_datetime']]

In [ ]:
sbp_df = sbp_df.rename(columns = {'value_as_number':'sbp'})
dbp_df = dbp_df.rename(columns = {'value_as_number':'dbp'})
sbp_df['measurement_datetime'] = sbp_df['measurement_datetime'].apply(lambda x: str(x).split('+')[0])
sbp_df['measurement_datetime'] = sbp_df['measurement_datetime'].apply(lambda x: str(x).split(' ')[0])
sbp_df['measurement_datetime'] = pd.to_datetime(sbp_df['measurement_datetime'],format='mixed')
sbp_df = sbp_df.reset_index(drop=True)
dbp_df['measurement_datetime'] = dbp_df['measurement_datetime'].apply(lambda x: str(x).split('+')[0])
dbp_df['measurement_datetime'] = dbp_df['measurement_datetime'].apply(lambda x: str(x).split(' ')[0])
dbp_df['measurement_datetime'] = pd.to_datetime(dbp_df['measurement_datetime'],format='mixed')
dbp_df = dbp_df.reset_index(drop=True)
del bloodp_df

In [ ]:
len(sbp_df),len(dbp_df)

In [ ]:
#do not run
values_df = {}
values_df['glucose'] = hba1c_df_e
values_df['cholesterol'] = cholesterol_df_e
values_df['sbp'] = sbp_df
values_df['dbp'] = dbp_df

In [ ]:
#do not run
results = {}
windows = {
            "±6 months": 182.62,
            "±1 year": 365.24,
            "±2 years": 2*365.24,
            "±3 years": 3*3365.24
        }
for kk in ['glucose','cholesterol','bp']:
    df_dates = dfs_survey[f'{kk}_filtered'].copy()
    if kk!='bp':
        valuedf = values_df[f'{kk}'].copy()

        df_dates['ref_date'] = pd.to_datetime(df_dates['survey_datetime1'],format = 'mixed')
        df = df_dates.merge(valuedf, on='person_id', how='left')
        df['diff_days'] = (
            df['measurement_datetime'] - df['ref_date']
        ).dt.days.abs()

        for name, days in windows.items():
            valid = df[df['diff_days'] <= days]
            n_people = valid['person_id'].nunique()
            results[f'{kk}_{name}'] = n_people
    else:
        for kk1 in ['sbp','dbp']:
            valuedf = values_df[f'{kk1}'].copy()

            df_dates['ref_date'] = pd.to_datetime(df_dates['survey_datetime1'],format = 'mixed')
            df = df_dates.merge(valuedf, on='person_id', how='left')
            df['diff_days'] = (
                df['measurement_datetime'] - df['ref_date']
            ).dt.days.abs()

            for name, days in windows.items():
                valid = df[df['diff_days'] <= days]
                n_people = valid['person_id'].nunique()
                results[f'{kk1}_{name}'] = n_people
    del df
    import gc
    gc.collect()

In [ ]:
#do not run
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


# ---- Convert to DataFrame ----
df = pd.Series(results).reset_index()
df.columns = ['key', 'count']

df[['metric', 'window']] = df['key'].str.split('_', expand=True)

# ---- Pivot ----
pivot = df.pivot(index='metric', columns='window', values='count')

# Ensure order
window_order = ['±6 months', '±1 year', '±2 years', '±3 years']
pivot = pivot[window_order]

# ---- Plot ----
x = np.arange(len(pivot.index))
width = 0.2

plt.figure(figsize = (15,5))

for i, w in enumerate(window_order):
    bars = plt.bar(x + i*width - 1.5*width, pivot[w], width, label=w)
    
    # annotate
    for bar in bars:
        h = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, h, f'{int(h):,}',
                 ha='center', va='bottom', fontsize=10)

plt.xticks(x, pivot.index)
plt.xlabel("Measurement")
plt.ylabel("People with Valid Measurement")
# plt.title("Data Availability vs Time Window")
plt.legend()
plt.grid(axis='y')

plt.show()


In [ ]:

def bp_from_lab(sbp, dbp, yes_score):
    """
    Blood pressure scoring:
    - SBP <130 and DBP <80  -> 7 points
    - SBP 130–139 or DBP 80–89 -> 3 points
    - SBP >=140 or DBP >=90 -> 0/1 points
    """


    if pd.isna(sbp) or pd.isna(dbp):
        return 'no close value to match'

    # 1. HIGH first
    if sbp >= 140 or dbp >= 90:
        return yes_score

    # 2. NORMAL
    if sbp < 130 and dbp < 80:
        return 7

    # 3. INTERMEDIATE
    if (130 <= sbp < 140) or (80 <= dbp < 90):
        return 3

    return 0


def blood_pressure_score_old(person_id, survey_df, sbp = None, dbp = None, bpdf = None, window_days = None):

    rows1 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"] == codes_t['bp']['q1'])
    ]
    rows2 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"] == codes_t['bp']['q2'])
    ]
    rows3 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"] == codes_t['bp']['q3'])
    ]

    ans1 = latest_valid_answer(rows1,['PMI: Skip'])
    
    # ---- Q1 ----
    if ans1 is None:
        return 'exclude', 'not needed', 'not needed', 'exclude'

    if 'self' not in ans1.lower():
        return 7, 'not needed', 'not needed', 7

    # ---- Q2 & Q3 ----
    def score_followup_bp(rows):
        ans = latest_valid_answer(rows,['PMI: Skip'])
        if ans is None:
            return None

        if 'no' in ans.lower():
            score = bp_from_lab(sbp, dbp, yes_score=0)
        else:
            score = bp_from_lab(sbp, dbp, yes_score=1)

        return score

    def score_followup_bp_df(rows, bpdf, person_id, window_days):
        ans_row = latest_valid_row(rows, ['PMI: Skip'])
        if ans_row is None:
            return None

        ans = ans_row["answer"]
        q_date = ans_row["survey_datetime"]



        val_u = closest_value_in_window(
            bpdf,
            person_id,
            q_date,
            days=window_days
        )
        if val_u is not None:
            sbp, dbp = val_u['sbp'],val_u['dbp']
        else:
            sbp, dbp = None,None

        if 'no' in ans.lower():
            score = bp_from_lab(sbp, dbp, yes_score=0)
        else:
            score = bp_from_lab(sbp, dbp, yes_score=1)
            
        return score if isinstance(score, (int, float)) else 'exclude'
    
    if bpdf is None:
        a2 = score_followup_bp(rows2)
        a3 = score_followup_bp(rows3)
    else:
        a2 = score_followup_bp_df(rows2, bpdf, person_id, window_days=window_days)
        a3 = score_followup_bp_df(rows3, bpdf, person_id, window_days=window_days)

    numeric_vals = [v for v in (a2, a3) if isinstance(v, (int, float))]

    if numeric_vals:
        final_answer = max(numeric_vals)
    else:
        final_answer = 'exclude'

    return 'go to q2,q3', a2, a3, final_answer


def blood_pressure_score(person_id, survey_df, sybp_df, dibp_df, window_days):

    rows1 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"] == codes_t['bp']['q1'])
    ]
    rows2 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"] == codes_t['bp']['q2'])
    ]
    rows3 = survey_df[
        (survey_df["person_id"] == person_id) &
        (survey_df["question_concept_id"] == codes_t['bp']['q3'])
    ]

    ans1 = latest_valid_answer(rows1,['PMI: Skip'])
    

    # ---- Q1 ----
    if ans1 is None:
        return 'exclude', 'not needed', 'not needed', 'exclude'

    if 'self' not in ans1.lower():
        return 7, 'not needed', 'not needed', 7

    # ---- Q2 & Q3 ----
    def score_followup_bp(rows, sbpdf, dbpdf, person_id, window_days):
        ans_row = latest_valid_row(rows, ['PMI: Skip'])
        if ans_row is None:
            return 'exclude',None

        ans = ans_row["answer"]
        q_date = ans_row["survey_datetime"]

        q_date = pd.to_datetime(str(q_date).split(' ')[0])

        val_su = closest_value_in_window(
            sbpdf,
            person_id,
            q_date,
            days=window_days
        )
        if val_su is not None:
            sbp = val_su['sbp']
        else:
            sbp = None
        
        val_du = closest_value_in_window(
            dbpdf,
            person_id,
            q_date,
            days=window_days
        )
        if val_du is not None:
            dbp = val_du['dbp']
        else:
            dbp = None
            
        if 'no' in ans.lower():
            score = bp_from_lab(sbp, dbp, yes_score=0)
        else:
            score = bp_from_lab(sbp, dbp, yes_score=1)
        
        
        return ans, score
        
    ans2, a2 = score_followup_bp(rows2, sybp_df, dibp_df, person_id, window_days=window_days)
    ans3, a3 = score_followup_bp(rows3, sybp_df, dibp_df, person_id, window_days=window_days)

    numeric_vals = [v for v in [a2, a3] if isinstance(v, (int, float))]

    if numeric_vals:
        final_answer = max(numeric_vals)
    else:
        final_answer = 'no close value to match'
        

    return 'go to q2,q3', a2, a3, final_answer
       
pid_q = 2765479
# pid_q = 1449307  
# pid_q = 1153605 
# pid_q = 2616415
# pid_q = 1857421
# pid_q = 1277645
pid_q = 1389583
blood_pressure_score_old(pid_q, dfs_survey['bp'],120,80,None),\
blood_pressure_score_old(pid_q, dfs_survey['bp'],np.nan,80,None),\
blood_pressure_score_old(pid_q, dfs_survey['bp'],150,80,None),\
blood_pressure_score(pid_q, dfs_survey['bp'],sbp_df, dbp_df, 365.24)

In [ ]:
import gc
gc.collect()

In [ ]:
# def compute_bcs_score(person_id, survey_df,hba1c,sbp,dbp,cl):
def compute_bcs_score(person_id, survey_df,hba1c_df_eo,sbp_df_o,dbp_df_o,cholesterol_df_eo,days):
    scores = {}
    filtered = '_filtered'
#     filtered = ''
    scores["smoking"] = smoking_score(person_id, survey_df['smoking'+filtered])[-1]
    scores["alcohol"] = alcohol_score(person_id, survey_df['alcohol'+filtered])
    scores["pa_vigorous"] = physical_activity_vigorous_score(person_id, survey_df['vig_pa'+filtered])[-1]
    scores["pa_moderate"] = physical_activity_moderate_score(person_id, survey_df['mod_pa'+filtered])[-1]
    scores["sleep"] = sleep_score(person_id, survey_df['sleep'+filtered])[-1]
    scores["stress"] = stress_score(person_id, survey_df['stress'+filtered])
    scores["social"] = social_engagement_score(person_id, survey_df['social'+filtered])
    scores["meaning"] = meaning_in_life_score(person_id, survey_df['meaning_in_life'+filtered])
    scores["hearing"] = hearing_impairment_score(person_id, survey_df['hearing'+filtered])
#     scores["blood_pressure"] = blood_pressure_score_old(person_id, survey_df['bp'],sbp,dbp)[-1]
    scores["blood_pressure"] = blood_pressure_score(person_id, survey_df['bp'+filtered],sbp_df_o, dbp_df_o,days)[-1]
#     scores["glucose_1"] = glucose_score_old(person_id, survey_df['glucose'], 'q3','q4', hba1c)[-1]
#     scores["glucose_2"] = glucose_score_old(person_id, survey_df['glucose'], 'q1','q2',hba1c)[-1]
#     scores["glucose_unk"] = glucose_score_old(person_id, survey_df['glucose'], 'q5','q6', hba1c)[-1]
    scores["glucose_1"] = glucose_score(person_id, survey_df['glucose'+filtered], 'q3','q4', hba1c_df_eo,days)[-1]
    scores["glucose_2"] = glucose_score(person_id, survey_df['glucose'+filtered], 'q1','q2', hba1c_df_eo,days)[-1]
    scores["glucose_unk"] = glucose_score(person_id, survey_df['glucose'+filtered], 'q5','q6', hba1c_df_eo,days)[-1]
    
    vals = [scores["glucose_1"],scores["glucose_2"],scores["glucose_unk"]]
    numeric_vals = [v for v in vals if isinstance(v, (int, float))]
    if numeric_vals:
        scores['glucose']= max(numeric_vals) #verify logic
    elif 'exclude' in vals:
        scores["glucose"] = 'exclude'
    else:
        scores["glucose"] = 'no close value to match'
        
    
    
#     scores["cholesterol"] = cholesterol_score_old(person_id, survey_df['cholesterol'],cl)[-1]
    scores["cholesterol"] = cholesterol_score(person_id, survey_df['cholesterol'+filtered],cholesterol_df_eo,days)[-1]
    
    
    total_score = sum(
        v for k, v in scores.items()
        if k not in ['glucose_1', 'glucose_2', 'glucose_unk']
        and isinstance(v, (int, float))
    )
    
    

    
    scores['person_id'] = person_id
    scores['total_score'] = total_score
    return pd.DataFrame(scores,['']).set_index('person_id').reset_index()



# # df  = compute_bcs_score(2765479, dfs_survey ,6.6,120,90,190)
# # df  = compute_bcs_score(1449307, dfs_survey ,6.6,120,90,190)
# # df  = compute_bcs_score(1153605, dfs_survey ,6.6,120,90,190)
# # df  = compute_bcs_score(2616415, dfs_survey ,6.6,120,90,190)
# # df  = compute_bcs_score(1857421, dfs_survey ,6.6,120,90,190)
# # df  = compute_bcs_score(1277645, dfs_survey ,6.6,120,90,190)

df  = compute_bcs_score(2765479, dfs_survey ,hba1c_df_e,sbp_df,dbp_df,cholesterol_df_e,3*365.24)
df  = compute_bcs_score(1449307, dfs_survey ,hba1c_df_e,sbp_df,dbp_df,cholesterol_df_e,3*365.24)
df  = compute_bcs_score(1153605, dfs_survey ,hba1c_df_e,sbp_df,dbp_df,cholesterol_df_e,3*365.24)
df  = compute_bcs_score(2616415, dfs_survey ,hba1c_df_e,sbp_df,dbp_df,cholesterol_df_e,3*365.24)
df  = compute_bcs_score(1857421, dfs_survey ,hba1c_df_e,sbp_df,dbp_df,cholesterol_df_e,3*365.24)
df  = compute_bcs_score(1277645, dfs_survey ,hba1c_df_e,sbp_df,dbp_df,cholesterol_df_e,3*365.24)




In [ ]:
# all_ty = sorted(list(dfs_survey.keys()))
all_ty = sorted([i for i in sorted(list(dfs_survey.keys())) if 'filtered' in i])
set_i = set(dfs_survey[all_ty[0]]['person_id'])
for i in all_ty[1:]:
    set_j = set_i.union(set(dfs_survey[i]['person_id']))
    set_i = set_j
all_people = sorted(list(set_j))

In [ ]:
len(all_people)

In [ ]:
# import time
# st = time.time()
# dfs = []
# for kk in all_people[0:500]:
# #     dfs.append(compute_bcs_score(kk, dfs_survey ,6.6,120,90,190))
#     dfs.append(compute_bcs_score(kk, dfs_survey ,hba1c_df_e,sbp_df,dbp_df,cholesterol_df_e,365.24))
# dfs_all = pd.concat(dfs,ignore_index = True)
# ed = time.time()
# dfs_all

In [ ]:
gc.collect()

In [ ]:
from multiprocessing import Pool
from tqdm import tqdm
import pandas as pd
import time

# ---- CONFIG ----
CHUNK_SIZE = 1000
OUTPUT_FILE = "bcs_031826.csv"
N_PROCESSES = 28 

# ---- GLOBAL STORAGE ----
GLOBAL = {}

def init_worker(survey, hba1c, sbp, dbp, chol):
    GLOBAL["survey"] = survey
    GLOBAL["hba1c"] = hba1c
    GLOBAL["sbp"] = sbp
    GLOBAL["dbp"] = dbp
    GLOBAL["chol"] = chol

# ---- WORKER ----
def wrapper(kk):
    return compute_bcs_score(
        kk,
        GLOBAL["survey"],
        GLOBAL["hba1c"],
        GLOBAL["sbp"],
        GLOBAL["dbp"],
        GLOBAL["chol"],
        3 * 365.24
    )

# ---- MAIN ----
if __name__ == "__main__":

    st = time.time()

    dfs_chunk = []
    first_write = True

    with Pool(
        processes=N_PROCESSES,
        initializer=init_worker,
        initargs=(dfs_survey, hba1c_df_e, sbp_df, dbp_df, cholesterol_df_e)
    ) as pool:

        for i, result in enumerate(
            tqdm(pool.imap_unordered(wrapper, all_people), total=len(all_people))
        ):
            dfs_chunk.append(result)

            # ---- SAVE EVERY CHUNK ----
            if (i + 1) % CHUNK_SIZE == 0:
                df_out = pd.concat(dfs_chunk, ignore_index=True)

                df_out.to_csv(
                    OUTPUT_FILE,
                    mode='w' if first_write else 'a',
                    header=first_write,
                    index=False
                )

                print(f"Saved {i+1} people")

                dfs_chunk = []
                first_write = False

    # ---- SAVE REMAINING ----
    if dfs_chunk:
        df_out = pd.concat(dfs_chunk, ignore_index=True)
        df_out.to_csv(
            OUTPUT_FILE,
            mode='w' if first_write else 'a',
            header=first_write,
            index=False
        )

    ed = time.time()
    print(f"Time: {ed - st:.2f}s")


In [ ]:
len(set(sbp_df['person_id']).intersection(set(dbp_df['person_id'])))

In [ ]:
len(set(cholesterol_df_e['person_id']))

In [ ]:
len(set(hba1c_df_e['person_id']))

In [ ]:
cols = ['smoking', 'alcohol', 'pa_vigorous', 'pa_moderate',
       'sleep', 'stress', 'social', 'meaning', 'hearing', 'blood_pressure',
       'glucose', 'cholesterol']
cols_n = ['smoking', 'alcohol', 'pa_vigorous', 'pa_moderate',
       'sleep', 'stress', 'social', 'meaning', 'hearing']
cols_v = ['blood_pressure',
       'glucose', 'cholesterol']

bad_vals = ['exclude']
bad_vals_s = ['exclude', 'no close value to match']

len(dfs_all[dfs_all[cols].isin(bad_vals_s).all(axis=1)]),\
len(dfs_all)- len(dfs_all[dfs_all[cols].isin(bad_vals_s).all(axis=1)]),\
len(dfs_all) - len(dfs_all[dfs_all[cols_v].isin(bad_vals_s).all(axis=1)]),\
len(dfs_all[~dfs_all[cols].isin(bad_vals_s).any(axis=1)]),\
len(dfs_all[dfs_all[cols].isin(bad_vals).all(axis=1)]),\
len(dfs_all)- len(dfs_all[dfs_all[cols].isin(bad_vals).all(axis=1)]),\
len(dfs_all) - len(dfs_all[dfs_all[cols_v].isin(bad_vals).all(axis=1)]),\
len(dfs_all[~dfs_all[cols].isin(bad_vals).any(axis=1)])


In [ ]:
# len(dfs_all[dfs_all[cols].isin(bad_vals).any(axis=1)]),\
# len(dfs_all[
#     ~dfs_all[cols].isin(bad_vals).all(axis=1)
# ]),\
# len(dfs_all[
#     ~dfs_all[cols_n].isin(bad_vals).all(axis=1)
# ])

### MDD harmonize

In [ ]:
import pandas as pd
import os
exclusion_snomed_concept_ids = [35622934, 35624743, 35624744, 35624745, 35624747, 35624748, \
37109940, 37117177, 372599, 37312578, 4001733, 4008566, 4009648, 4028027, 4030856, 4037669, \
4045263, 4071442, 4085662, 4100366, 4102603, 4105330, 4131027, 4133495, 4141603, 4144519, \
4148842, 4148934, 4150985, 4154283, 4155798, 4161200, 4166701, 4168389, 4172156, 4177651, \
4185096, 4194222, 4195158, 4197669, 4200385, 4201739, 4215917, 4217940, 4220617, 4220618, \
4244078, 4262111, 4262272, 4280361, 42872412, 42872413, 4287544, 43020451, 4307804, 4307956, \
4310121, 4310821, 432290, 432299, 432300, 4324945, 432597, 432598, 4327669, 432865, 432866, \
432876, 433442, 433443, 433450, 433734, 433742, 433743, 433990, 433992, 433996, 434321, 434332, \
 434901, 435217, 435218, 435219, 435225, 435226, 435230, 435235, 435236, 435782, 435783, 436067, \
 436071, 436072, 436086, 436384, 436385, 436386, 436665, 436673, 436944, 437243, 437250, 437528, 437529, \
438724, 439001, 439004, 439245, 439246, 439248, 439249, 439250, 439251, 439253, 439254, 439255, \
439256, 439274, 439275, 439785, 440067, 440077, 440078, 440079, 440368, 440373, 440686, 441538,\
441828, 441834, 441835, 441836, 442570, 442600, 443797, 443906, 444396]
exclusion_icd_concept_ids = [1568206, 1568207, 1568211, 1568213, 1568214, 1595540, 3229343, 35207126, \
35207127, 35207128, 35207129, 35207130, 35207131, 35207136, 35207141, 35207145, 35207146, 35207147, \
35207148, 35207149, 44819538, 44819539, 44819542, 44820712, 44820714, 44820718, 44820719, 44820720, \
44820721, 44820722, 44821815, 44821816, 44821817, 44821819, 44822969, 44822970, 44822971, 44822972, \
44822973, 44822977, 44822978, 44822979, 44822980, 44822981, 44824109, 44824112, 44824113, 44824115, \
44824116, 44824117, 44824118, 44825288, 44825290, 44825291, 44826491, 44826492, 44826493, 44826494, \
44826495, 44826496, 44826497, 44826499, 44826500, 44827646, 44827647, 44827649, 44827650, 44827657, \
44828825, 44828826, 44828830, 44828831, 44828832, 44829915, 44829919, 44829921, 44829922, 44829924, \
44829925, 44831084, 44831085, 44831086, 44831087, 44831091, 44831092, 44832223, 44832224, 44832225, \
44832226, 44832230, 44832233, 44832234, 44832235, 44832236, 44832238, 44833399, 44833400, 44834588, \
44834590, 44835779, 44835780, 44835781, 44835786, 44835788, 44836962, 44836963, 44836964, 44836965, \
44836966, 44836969, 45538061, 45538062, 45538063, 45538064, 45542841, 45542842, 45542843, 45542844, \
45547696, 45547703, 45552477, 45557201, 45557208, 45562038, 45566839, 45566840, 45566841, 45571753, \
45571756, 45576538, 45576539, 45576540, 45591134, 45591135, 45591136, 45595903, 45595910, 45926881]
dataset_condition_sql = f"""
SELECT
    c.person_id,
    c.condition_concept_id,
    cc.concept_name AS standard_concept_name,
    cc.concept_code AS standard_concept_code,
    cc.vocabulary_id AS standard_vocabulary,
    c.condition_source_concept_id,
    cs.concept_name AS source_concept_name,
    cs.concept_code AS source_concept_code,
    cs.vocabulary_id AS source_vocabulary,
    c.condition_start_datetime,
    c.condition_end_datetime
FROM `{os.environ["WORKSPACE_CDR"]}.condition_occurrence` c
LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` cc
    ON c.condition_concept_id = cc.concept_id
LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` cs
    ON c.condition_source_concept_id = cs.concept_id
WHERE c.condition_concept_id IN UNNEST({exclusion_snomed_concept_ids})
   OR c.condition_source_concept_id IN UNNEST({exclusion_icd_concept_ids})
"""

both_exclude_df_all = pd.read_gbq(
    dataset_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
)


dataset_observation_sql = f"""
    SELECT
        person_id,
        o_standard_concept.concept_code as standard_concept_code,
        o_standard_concept.vocabulary_id as standard_vocabulary,
        o_source_concept.concept_name as source_concept_name,
        o_source_concept.concept_code as source_concept_code,
        o_source_concept.vocabulary_id as source_vocabulary,
        observation.unit_source_value,
        observation.qualifier_source_value,
        observation.value_source_concept_id,
        observation.value_source_value,
        observation.questionnaire_response_id 
    FROM
        ( SELECT
            * 
        FROM
            `{os.environ["WORKSPACE_CDR"]}.observation` observation 
        WHERE
            (
                observation_source_concept_id IN UNNEST({exclusion_icd_concept_ids})
            )) observation 
    LEFT JOIN
        `{os.environ["WORKSPACE_CDR"]}.concept` o_standard_concept 
            ON observation.observation_concept_id = o_standard_concept.concept_id 
    LEFT JOIN
        `{os.environ["WORKSPACE_CDR"]}.concept` o_source_concept 
            ON observation.observation_source_concept_id = o_source_concept.concept_id"""

icd_observation_df = pd.read_gbq(
    dataset_observation_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
)
both_exclude_df_all = pd.concat([both_exclude_df_all,icd_observation_df],ignore_index = True)




In [ ]:
# Late MDD by both
snomed_concept_ids = [4152280,4282096,4282316,432285,432883,438998,434911,438406,4298317,\
439259,4094358,35615153,35615155,35615152,4324959,4205471,4031328,45757196,\
4093584,35615154,4304140]
icd_concept_ids = [35207155,1568218,35207157,44825293,1568219,44821821,35207161,35207156,\
44831090,44831089,35207158,45552479,44822975,35207151,45571761,35207150,\
44835785,44829923,35207152,45547704,44834589,45547706,44827654,44827656,\
45576541,35207159,44824114,44827655,44822976,44835784,35207153,44832232,44820717,\
1568217, 35207154, 35207160,35207162, 35207163, \
35207164, 35207165, 35207166, 37200319, 37200320, \
37200321, 37200322, 766349]

dataset_13847333_person_sql = """
    SELECT
        * 
    FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
        """

df_person = pd.read_gbq(
    dataset_13847333_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_condition_sql = f"""
SELECT
    c.person_id,
    c.condition_concept_id,
    cc.concept_name AS standard_concept_name,
    cc.concept_code AS standard_concept_code,
    cc.vocabulary_id AS standard_vocabulary,
    c.condition_source_concept_id,
    cs.concept_name AS source_concept_name,
    cs.concept_code AS source_concept_code,
    cs.vocabulary_id AS source_vocabulary,
    c.condition_start_datetime,
    c.condition_end_datetime
FROM `{os.environ["WORKSPACE_CDR"]}.condition_occurrence` c
LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` cc
    ON c.condition_concept_id = cc.concept_id
LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` cs
    ON c.condition_source_concept_id = cs.concept_id
WHERE c.condition_concept_id IN UNNEST({snomed_concept_ids})
   OR c.condition_source_concept_id IN UNNEST({icd_concept_ids})
"""

both_df_all = pd.read_gbq(
    dataset_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
)

dataset_observation_sql = f"""
    SELECT
        person_id,
        o_standard_concept.concept_code as standard_concept_code,
        o_standard_concept.vocabulary_id as standard_vocabulary,
        o_source_concept.concept_name as source_concept_name,
        o_source_concept.concept_code as source_concept_code,
        o_source_concept.vocabulary_id as source_vocabulary,
        observation.unit_source_value,
        observation.qualifier_source_value,
        observation.value_source_concept_id,
        observation.value_source_value,
        observation.questionnaire_response_id 
    FROM
        ( SELECT
            * 
        FROM
            `{os.environ["WORKSPACE_CDR"]}.observation` observation 
        WHERE
            (
                observation_source_concept_id IN UNNEST({icd_concept_ids})
            )) observation 
    LEFT JOIN
        `{os.environ["WORKSPACE_CDR"]}.concept` o_standard_concept 
            ON observation.observation_concept_id = o_standard_concept.concept_id 
    LEFT JOIN
        `{os.environ["WORKSPACE_CDR"]}.concept` o_source_concept 
            ON observation.observation_source_concept_id = o_source_concept.concept_id"""

icd_observation_df = pd.read_gbq(
    dataset_observation_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
)
both_df_all = pd.concat([both_df_all,icd_observation_df],ignore_index = True)


both_df_all_mdd = both_df_all.copy()

both_df_all = df_person[['person_id','dob']].merge(both_df_all,on = 'person_id')

both_df_all['dob'] = pd.to_datetime(both_df_all['dob'], utc=True).dt.tz_localize(None)
both_df_all['condition_start_datetime'] = pd.to_datetime(both_df_all['condition_start_datetime'], errors='coerce', utc=True).dt.tz_localize(None)
both_df_all['condition_end_datetime'] = pd.to_datetime(both_df_all['condition_end_datetime'], errors='coerce', utc=True).dt.tz_localize(None)

both_df_all['diagnosis_age'] = (both_df_all['condition_start_datetime'] - both_df_all['dob']).dt.days / 365.25

mask = both_df_all.groupby('person_id')['diagnosis_age'].transform(lambda x: (x < 60).any())
late_both_df_all = both_df_all[~mask]

len(set(both_df_all['person_id'])),len(set(late_both_df_all['person_id']))

In [ ]:
cases_drug_sql = """
    SELECT
        d_exposure.person_id,
        d_exposure.drug_concept_id,
        d_standard_concept.concept_name as standard_concept_name,
        d_standard_concept.concept_code as standard_concept_code,
        d_standard_concept.vocabulary_id as standard_vocabulary,
        d_exposure.drug_exposure_start_datetime,
        d_exposure.drug_exposure_end_datetime,
        d_exposure.verbatim_end_date,
        d_exposure.drug_type_concept_id,
        d_type.concept_name as drug_type_concept_name,
        d_exposure.stop_reason,
        d_exposure.refills,
        d_exposure.quantity,
        d_exposure.days_supply,
        d_exposure.sig,
        d_exposure.route_concept_id,
        d_route.concept_name as route_concept_name,
        d_exposure.lot_number,
        d_exposure.visit_occurrence_id,
        d_visit.concept_name as visit_occurrence_concept_name,
        d_exposure.drug_source_value,
        d_exposure.drug_source_concept_id,
        d_source_concept.concept_name as source_concept_name,
        d_source_concept.concept_code as source_concept_code,
        d_source_concept.vocabulary_id as source_vocabulary,
        d_exposure.route_source_value,
        d_exposure.dose_unit_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.drug_exposure` d_exposure 
        WHERE
            (
                drug_concept_id IN (SELECT
                    DISTINCT ca.descendant_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                JOIN
                    (SELECT
                        DISTINCT c.concept_id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id             
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                        WHERE
                            concept_id IN (19052903, 19056379, 19057607, 19071905, 19113039, 19113040, 19124477, 19127669, 19136626, 21604555, 35602825, 42899307, 700299, 704984, 733008, 735979, 740275, 745466, 745790, 751246, 752061, 756018, 757688, 766529, 766814, 767410, 785788, 794852, 800878)             
                            AND full_text LIKE '%_rank1]%'       ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) b 
                        ON (ca.ancestor_id = b.concept_id)))) d_exposure 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_standard_concept 
                ON d_exposure.drug_concept_id = d_standard_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_type 
                ON d_exposure.drug_type_concept_id = d_type.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_route 
                ON d_exposure.route_concept_id = d_route.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                ON d_exposure.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_visit 
                ON v.visit_concept_id = d_visit.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_source_concept 
                ON d_exposure.drug_source_concept_id = d_source_concept.concept_id"""

cases_drug_df = pd.read_gbq(
    cases_drug_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

cases_drug_df.head(5)

In [ ]:
late_both_df_all_consider = late_both_df_all[~late_both_df_all['person_id'].isin(\
                            set(both_exclude_df_all['person_id']))]

len(set(late_both_df_all_consider['person_id'])),len(set(late_both_df_all['person_id']))

In [ ]:
late_both_df_all_final = late_both_df_all_consider[~late_both_df_all_consider['person_id'].isin(\
                            set(cases_drug_df['person_id']))]

len(set(late_both_df_all_final['person_id'])),len(set(late_both_df_all_consider['person_id']))

In [ ]:
import pandas as pd
import os

controls_drug_sql = """
    SELECT
        d_exposure.person_id,
        d_exposure.drug_concept_id,
        d_standard_concept.concept_name as standard_concept_name,
        d_standard_concept.concept_code as standard_concept_code,
        d_standard_concept.vocabulary_id as standard_vocabulary,
        d_exposure.drug_exposure_start_datetime,
        d_exposure.drug_exposure_end_datetime,
        d_exposure.verbatim_end_date,
        d_exposure.drug_type_concept_id,
        d_type.concept_name as drug_type_concept_name,
        d_exposure.stop_reason,
        d_exposure.refills,
        d_exposure.quantity,
        d_exposure.days_supply,
        d_exposure.sig,
        d_exposure.route_concept_id,
        d_route.concept_name as route_concept_name,
        d_exposure.lot_number,
        d_exposure.visit_occurrence_id,
        d_visit.concept_name as visit_occurrence_concept_name,
        d_exposure.drug_source_value,
        d_exposure.drug_source_concept_id,
        d_source_concept.concept_name as source_concept_name,
        d_source_concept.concept_code as source_concept_code,
        d_source_concept.vocabulary_id as source_vocabulary,
        d_exposure.route_source_value,
        d_exposure.dose_unit_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.drug_exposure` d_exposure 
        WHERE
            (
                drug_concept_id IN (SELECT
                    DISTINCT ca.descendant_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                JOIN
                    (SELECT
                        DISTINCT c.concept_id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id             
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                        WHERE
                            concept_id IN (1398039, 1704139, 19006186, 19049402, 703470, 703547, 704984, 705755, 710062, 715259, 715939, 721724, 722031, 725131, 733008, 733896, 738156, 739138, 743670, 751412, 755695, 756018, 778268, 781705, 797617, 798834, 990678)             
                            AND full_text LIKE '%_rank1]%'       ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) b 
                        ON (ca.ancestor_id = b.concept_id)))) d_exposure 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_standard_concept 
                ON d_exposure.drug_concept_id = d_standard_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_type 
                ON d_exposure.drug_type_concept_id = d_type.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_route 
                ON d_exposure.route_concept_id = d_route.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                ON d_exposure.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_visit 
                ON v.visit_concept_id = d_visit.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_source_concept 
                ON d_exposure.drug_source_concept_id = d_source_concept.concept_id"""

controls_drug_df = pd.read_gbq(
    controls_drug_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

controls_drug_df.head(5)

### Harmonized meta cohort

In [ ]:
#stroke,dementia
snomed_concept_ids = [443454,43530683,762933,4111714,4108356,45772786,4110190,46273649,\
46270031,43531607,4110192,45767658,44782773,4046362,4077086,4045736,4146185,\
437544,437540,381591,4111716,4112023,4111717,437847]
icd_concept_ids = [44835952,44823042,44834737,44830090,1569193,35207819,1569210,\
45581782,37200507,37200499,37200509,37200506,37200508,37200498,\
1569199,1569212,1569208,45552804,45572097,45552805,45586594,45548030,\
45557551,1569209,45601045,45605806,1569211,1569197,45543190,45596212,\
45601044,45562367,45548029,45538395,45596214,45576884,45572099,45552803,\
45576889,45581783,45567187,45562364,1569198,1569205,45552801,37200503,\
37200497,37200502,37200504,37200496,1569196,1569207,1569203,45576886,\
45562363,45543189,45567188,45596213,45533462,1569204,45576888,45533463,\
1569206,1569194,45557553,45581781,45586593,45557552,45576887,45601042,\
45543188,45538394,45605807,45591474,45548028,45567189,45601041,45562362,\
1569195,1569215,45538396,37200511,37200501,37200513,37200510,37200512,\
37200500,1569202,1569217,1569213,45533464,45562365,45596215,45586595,\
45557555,45576885,1569214,45601047,45548027,1569216,1569200,45538397,\
45601043,45581784,45605809,45586596,45572098,45557554,45543187,45591476,\
45552806,45601046,45586597,45552802,45548026,1569201,35207821,44833577,\
44819712,44833576,44819713,44832384,44835950,44828982,35207820,1595598,\
1595597,44820786,44828897]
dataset_condition_sql = f"""
SELECT
    c.person_id,
    c.condition_concept_id,
    cc.concept_name AS standard_concept_name,
    cc.concept_code AS standard_concept_code,
    cc.vocabulary_id AS standard_vocabulary,
    c.condition_source_concept_id,
    cs.concept_name AS source_concept_name,
    cs.concept_code AS source_concept_code,
    cs.vocabulary_id AS source_vocabulary,
    c.condition_start_datetime,
    c.condition_end_datetime
FROM `{os.environ["WORKSPACE_CDR"]}.condition_occurrence` c
LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` cc
    ON c.condition_concept_id = cc.concept_id
LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` cs
    ON c.condition_source_concept_id = cs.concept_id
WHERE c.condition_concept_id IN UNNEST({snomed_concept_ids})
   OR c.condition_source_concept_id IN UNNEST({icd_concept_ids})
"""

both_df_all = pd.read_gbq(
    dataset_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
)


dataset_condition_sql = f"""
SELECT
    c.person_id,
    c.condition_concept_id,
    cc.concept_name AS standard_concept_name,
    cc.concept_code AS standard_concept_code,
    cc.vocabulary_id AS standard_vocabulary,
    c.condition_source_concept_id,
    cs.concept_name AS source_concept_name,
    cs.concept_code AS source_concept_code,
    cs.vocabulary_id AS source_vocabulary,
    c.condition_start_datetime,
    c.condition_end_datetime
FROM `{os.environ["WORKSPACE_CDR"]}.condition_occurrence` c
LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` cc
    ON c.condition_concept_id = cc.concept_id
LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` cs
    ON c.condition_source_concept_id = cs.concept_id
WHERE c.condition_concept_id IN UNNEST({snomed_concept_ids})
"""

snomed_df_all = pd.read_gbq(
    dataset_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
)


dataset_condition_sql = f"""
SELECT
    c.person_id,
    c.condition_concept_id,
    cc.concept_name AS standard_concept_name,
    cc.concept_code AS standard_concept_code,
    cc.vocabulary_id AS standard_vocabulary,
    c.condition_source_concept_id,
    cs.concept_name AS source_concept_name,
    cs.concept_code AS source_concept_code,
    cs.vocabulary_id AS source_vocabulary,
    c.condition_start_datetime,
    c.condition_end_datetime
FROM `{os.environ["WORKSPACE_CDR"]}.condition_occurrence` c
LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` cc
    ON c.condition_concept_id = cc.concept_id
LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` cs
    ON c.condition_source_concept_id = cs.concept_id
WHERE c.condition_source_concept_id IN UNNEST({icd_concept_ids})
"""

icd_df_all = pd.read_gbq(
    dataset_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
)

dataset_observation_sql = f"""
    SELECT
        person_id,
        o_standard_concept.concept_code as standard_concept_code,
        o_standard_concept.vocabulary_id as standard_vocabulary,
        o_source_concept.concept_name as source_concept_name,
        o_source_concept.concept_code as source_concept_code,
        o_source_concept.vocabulary_id as source_vocabulary,
        observation.unit_source_value,
        observation.qualifier_source_value,
        observation.value_source_concept_id,
        observation.value_source_value,
        observation.questionnaire_response_id 
    FROM
        ( SELECT
            * 
        FROM
            `{os.environ["WORKSPACE_CDR"]}.observation` observation 
        WHERE
            (
                observation_source_concept_id IN UNNEST({icd_concept_ids})
            )) observation 
    LEFT JOIN
        `{os.environ["WORKSPACE_CDR"]}.concept` o_standard_concept 
            ON observation.observation_concept_id = o_standard_concept.concept_id 
    LEFT JOIN
        `{os.environ["WORKSPACE_CDR"]}.concept` o_source_concept 
            ON observation.observation_source_concept_id = o_source_concept.concept_id"""

icd_observation_df = pd.read_gbq(
    dataset_observation_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
)
icd_df_all = pd.concat([icd_df_all,icd_observation_df],ignore_index = True)
both_df_all_stroke = pd.concat([both_df_all,icd_observation_df],ignore_index = True)

both_df_all_stroke['condition_start_datetime'] = pd.to_datetime(both_df_all_stroke['condition_start_datetime'], errors='coerce', utc=True).dt.tz_localize(None)
both_df_all_stroke['condition_end_datetime'] = pd.to_datetime(both_df_all_stroke['condition_end_datetime'], errors='coerce', utc=True).dt.tz_localize(None)



snomed_concept_ids = [35608576, 37018688, 37109056, 37110513, 374888, 375791, 376085, 376095, 376946, 377254, \
377527, 378125, 378419, 378726, 379778, 379784, 380701, 380986, 381832, 4009647, 4043378, \
4046090, 4047745, 4047747, 40483103, 4048875, 4092747, 4101137, 4139421, 4182210, 4196433, \
4218017, 4220313, 42538857, 4277444, 4278830, 43021816, 43530664, 43530666, 441002, 443605, \
443790, 443864, 444091, 44782763, 44782771, 762497]
icd_concept_ids = [1568088, 1568090, 1568295, 35207114, 35207337, 35207356, 35207357, 35207358, 35207359, 35207360,\
37402371, 37402372, 37402373, 37402374, 37402375, 37402377, 37402379, 37402380, 37402383,\
37402385, 37402388, 37402390, 37402393, 37402395, 37402398, 37402399, 37402400, 37402401, \
37402402, 37402403, 37402404, 37402406, 37402409, 37402410, 37402411, 37402412, 37402414, \
37402416, 37402417, 37402418, 37402419, 37402420, 37402422, 37402424, 37402425, 37402426, \
37402427, 37402429, 37402430, 37402431, 37402432, 37402433, 37402435, 37402437, 37402438, \
37402439, 37402440, 37402441, 37402443, 37402445, 37402446, 37402447, 37402448, 37402451, \
37402453, 37402454, 37402456, 44819534, 44820708, 44820709, 44821810, 44821811, 44821814, \
44824106, 44824152, 44826537, 44826538, 44827641, 44827644, 44829914, 44829915, 44829917, \
44831078, 44831079, 44831083, 44831122, 44832219, 44833435, 44834581, 44835772, 44835773, \
44835825, 44836954, 44836959, 45538000, 45538103, 45547730, 45591073, 45591076, 45595842, \
45595843, 45600684, 45605533]
dataset_condition_sql = f"""
SELECT
    c.person_id,
    c.condition_concept_id,
    cc.concept_name AS standard_concept_name,
    cc.concept_code AS standard_concept_code,
    cc.vocabulary_id AS standard_vocabulary,
    c.condition_source_concept_id,
    cs.concept_name AS source_concept_name,
    cs.concept_code AS source_concept_code,
    cs.vocabulary_id AS source_vocabulary,
    c.condition_start_datetime,
    c.condition_end_datetime
FROM `{os.environ["WORKSPACE_CDR"]}.condition_occurrence` c
LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` cc
    ON c.condition_concept_id = cc.concept_id
LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` cs
    ON c.condition_source_concept_id = cs.concept_id
WHERE c.condition_concept_id IN UNNEST({snomed_concept_ids})
   OR c.condition_source_concept_id IN UNNEST({icd_concept_ids})
"""

both_df_all = pd.read_gbq(
    dataset_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
)


dataset_condition_sql = f"""
SELECT
    c.person_id,
    c.condition_concept_id,
    cc.concept_name AS standard_concept_name,
    cc.concept_code AS standard_concept_code,
    cc.vocabulary_id AS standard_vocabulary,
    c.condition_source_concept_id,
    cs.concept_name AS source_concept_name,
    cs.concept_code AS source_concept_code,
    cs.vocabulary_id AS source_vocabulary,
    c.condition_start_datetime,
    c.condition_end_datetime
FROM `{os.environ["WORKSPACE_CDR"]}.condition_occurrence` c
LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` cc
    ON c.condition_concept_id = cc.concept_id
LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` cs
    ON c.condition_source_concept_id = cs.concept_id
WHERE c.condition_concept_id IN UNNEST({snomed_concept_ids})
"""

snomed_df_all = pd.read_gbq(
    dataset_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
)


dataset_condition_sql = f"""
SELECT
    c.person_id,
    c.condition_concept_id,
    cc.concept_name AS standard_concept_name,
    cc.concept_code AS standard_concept_code,
    cc.vocabulary_id AS standard_vocabulary,
    c.condition_source_concept_id,
    cs.concept_name AS source_concept_name,
    cs.concept_code AS source_concept_code,
    cs.vocabulary_id AS source_vocabulary,
    c.condition_start_datetime,
    c.condition_end_datetime
FROM `{os.environ["WORKSPACE_CDR"]}.condition_occurrence` c
LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` cc
    ON c.condition_concept_id = cc.concept_id
LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` cs
    ON c.condition_source_concept_id = cs.concept_id
WHERE c.condition_source_concept_id IN UNNEST({icd_concept_ids})
"""

icd_df_all = pd.read_gbq(
    dataset_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
)

dataset_observation_sql = f"""
    SELECT
        person_id,
        o_standard_concept.concept_code as standard_concept_code,
        o_standard_concept.vocabulary_id as standard_vocabulary,
        o_source_concept.concept_name as source_concept_name,
        o_source_concept.concept_code as source_concept_code,
        o_source_concept.vocabulary_id as source_vocabulary,
        observation.unit_source_value,
        observation.qualifier_source_value,
        observation.value_source_concept_id,
        observation.value_source_value,
        observation.questionnaire_response_id 
    FROM
        ( SELECT
            * 
        FROM
            `{os.environ["WORKSPACE_CDR"]}.observation` observation 
        WHERE
            (
                observation_source_concept_id IN UNNEST({icd_concept_ids})
            )) observation 
    LEFT JOIN
        `{os.environ["WORKSPACE_CDR"]}.concept` o_standard_concept 
            ON observation.observation_concept_id = o_standard_concept.concept_id 
    LEFT JOIN
        `{os.environ["WORKSPACE_CDR"]}.concept` o_source_concept 
            ON observation.observation_source_concept_id = o_source_concept.concept_id"""

icd_observation_df = pd.read_gbq(
    dataset_observation_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
)
icd_df_all = pd.concat([icd_df_all,icd_observation_df],ignore_index = True)
both_df_all_dementia = pd.concat([both_df_all,icd_observation_df],ignore_index = True)

both_df_all_dementia['condition_start_datetime'] = pd.to_datetime(both_df_all_dementia['condition_start_datetime'], errors='coerce', utc=True).dt.tz_localize(None)
both_df_all_dementia['condition_end_datetime'] = pd.to_datetime(both_df_all_dementia['condition_end_datetime'], errors='coerce', utc=True).dt.tz_localize(None)

both_df_all_meta = pd.concat([both_df_all_dementia,both_df_all_stroke,late_both_df_all_final],ignore_index = True)
len(set(both_df_all_meta['person_id']))
# len(set(snomed_df_all['person_id']).union(set(icd_df_all['person_id']))),\
# len(set(snomed_df_all['person_id'])),len(set(icd_df_all['person_id']))

In [ ]:
# Get unique IDs from each dataset
dementia_ids = set(both_df_all_dementia['person_id'])
stroke_ids = set(both_df_all_stroke['person_id'])
late_ids = set(late_both_df_all_final['person_id'])

only_dementia = dementia_ids - stroke_ids - late_ids
only_stroke = stroke_ids - dementia_ids - late_ids
only_late = late_ids - dementia_ids - stroke_ids

dementia_stroke = dementia_ids & stroke_ids
dementia_late = dementia_ids & late_ids
stroke_late = stroke_ids & late_ids
all_three = dementia_ids & stroke_ids & late_ids

summary = pd.DataFrame({
    'Group': [
        'Only dementia', 'Only stroke', 'Only late',
        'Dementia & Stroke', 'Dementia & Late', 'Stroke & Late', 'All three'
    ],
    'Count': [
        len(only_dementia), len(only_stroke), len(only_late),
        len(dementia_stroke - all_three),
        len(dementia_late - all_three),
        len(stroke_late - all_three),
        len(all_three)
    ]
})
summary

In [ ]:
dataset_67474743_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        person.self_reported_category_concept_id,
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) )"""

aou_ehr = pd.read_gbq(
    dataset_67474743_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

aou_ehr.head(5)



controls_meta = aou_ehr[~aou_ehr['person_id'].isin(set(both_df_all_meta['person_id']))]
print(len(set(controls_meta['person_id'])))
controls_meta = controls_meta[~controls_meta['person_id'].isin(set(both_df_all_mdd['person_id']))]
print(len(set(controls_meta['person_id'])))
controls_meta = controls_meta[~controls_meta['person_id'].isin(set(both_exclude_df_all['person_id']))]
print(len(set(controls_meta['person_id'])))
controls_meta = controls_meta[~controls_meta['person_id'].isin(set(controls_drug_df['person_id']))]
print(len(set(controls_meta['person_id'])))

In [ ]:
import pandas
import os

# This query represents dataset "ehr_fitbit_dataset" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_67474743_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        person.self_reported_category_concept_id,
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_fitbit = 1 ) )"""

fitbit_any = pandas.read_gbq(
    dataset_67474743_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

fitbit_any.head(5)


In [ ]:
import pandas as pd
bucket_ = os.environ.get('WORKSPACE_BUCKET')

df1 = pd.read_parquet(f'{bucket_}/second_level_ingested_data_stats_non_interpolated/stats_missingness_ehr.pq')
df2 = pd.read_parquet(f'{bucket_}/second_level_ingested_data_stats_non_interpolated/stats_missingness_without_ehr.pq')
sleep_df_ehr = pd.read_parquet(f'{bucket_}/sleep_ingestion_aggregated/sleep_summary_df_ehr_reingested.pq')
sleep_df_wehr = pd.read_parquet(f'{bucket_}/sleep_ingestion_aggregated/sleep_summary_df_no_ehr_reingested.pq')
df1['date'] = df1['date'].apply(lambda x: x.split(' ')[0])
df2['date'] = df2['date'].apply(lambda x: x.split(' ')[0])

df = pd.concat([df1,df2],ignore_index = True)
sleep_df = pd.concat([sleep_df_ehr,sleep_df_wehr],ignore_index = True)


sleep_df_ehr_eo = sleep_df_ehr[['person_id','date','time_asleep','time_in_bed']].merge(df1, on = ['person_id','date'])
sleep_df_wehr_eo = sleep_df_wehr[['person_id','date','time_asleep','time_in_bed']].merge(df2, on = ['person_id','date'])
sleep_dfg = pd.concat([sleep_df_ehr_eo, sleep_df_wehr_eo],ignore_index = True)
thres = 200
len(set(sleep_df_ehr_eo[(sleep_df_ehr_eo['hr_missingness']<thres)&(sleep_df_ehr_eo['steps_nw_missingness']<thres)&\
                  (sleep_df_ehr_eo['time_in_bed']>0)&(sleep_df_ehr_eo['time_in_bed']<=1440)]['person_id']))


In [ ]:
del df1, df2, sleep_df_wehr, sleep_df, sleep_df_ehr

In [ ]:
qc_fitbit = sleep_df_ehr_eo[(sleep_df_ehr_eo['hr_missingness']<thres)&(sleep_df_ehr_eo['steps_nw_missingness']<thres)&\
                  (sleep_df_ehr_eo['time_in_bed']>0)&(sleep_df_ehr_eo['time_in_bed']<=1440)]



In [ ]:
dataset_13847333_age_sql = """
    SELECT
        * 
    FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_fitbit = 1 """

df_age_f = pd.read_gbq(
    dataset_13847333_age_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")
df_age_f.head(5)


In [ ]:
len(set(qc_fitbit['person_id']))

In [ ]:
both_df_all_meta_qc_f = both_df_all_meta[both_df_all_meta['person_id'].isin(set(qc_fitbit['person_id']))]

control_all_meta_qc_f = controls_meta[controls_meta['person_id'].isin(set(qc_fitbit['person_id']))]

print(len(set(both_df_all_meta_qc_f['person_id'])),\
len(set(control_all_meta_qc_f['person_id'])))


In [ ]:
from collections import defaultdict
import numpy as np
import pandas as pd

results = defaultdict(list)


by_id = qc_fitbit.groupby('person_id')


for linker, frame in by_id:
    results['person_id'].append(linker)
    
    first_date = min(frame['date'])
    last_date = max(frame['date'])
    num_occurences = len(frame['date'])
    num_unique_dates = len(np.unique(frame['date']))
    
    results['first_date'].append(first_date)
    results['last_date'].append(last_date)
    results['num_occurences'].append(num_occurences)
    results['num_unique_dates'].append(num_unique_dates)
    
fitbit_hr_df = pd.DataFrame.from_dict(results)

print(len(fitbit_hr_df))

In [ ]:
results = defaultdict(list)

condition_columns_to_include = ['person_id', 'standard_concept_name', 'source_concept_code', 'condition_start_datetime', 'standard_vocabulary', 'condition_end_datetime', 'source_vocabulary']
columns = list(set(condition_columns_to_include)-set(['person_id']))
by_id = both_df_all_meta_qc_f.groupby('person_id')

for linker, frame in by_id:
    results['person_id'].append(linker)
    
    for c in columns:
        vals = frame[c].dropna().astype(str).values  # remove NaN / None
        if len(vals) == len(frame[c]):   # no missing values
            results.setdefault(c, []).append(np.unique(vals))
        else:
            results.setdefault(c, []).append(frame[c].values)
    
condition_df = pd.DataFrame.from_dict(results)

print(len(condition_df))

In [ ]:
fitbit_condition_df = pd.merge(fitbit_hr_df, condition_df[condition_columns_to_include], left_on='person_id',right_on='person_id', how='inner')
print(len(fitbit_condition_df))

fitbit_condition_df['condition_start_datetime_earliest'] = fitbit_condition_df['condition_start_datetime'].apply(lambda x: min(x))

fitbit_condition_df['condition_start_datetime_earliest'] = pd.to_datetime(
    fitbit_condition_df['condition_start_datetime_earliest'], errors='coerce'
)
fitbit_condition_df['first_date'] = pd.to_datetime(
    fitbit_condition_df['first_date'], errors='coerce'
)

fitbit_condition_df['condition_onset_minus_fitbit_start_days'] = (
    fitbit_condition_df['condition_start_datetime_earliest'] - fitbit_condition_df['first_date']
).dt.days

# fitbit_condition_df['condition_onset_minus_fitbit_start_days'] = (fitbit_condition_df['condition_start_datetime_earliest'].apply(lambda x: x.date())-fitbit_condition_df['first_date']).apply(lambda x: x.days)


In [ ]:
fitbit_condition_df

In [ ]:
import matplotlib.pyplot as plt

def create_data_histograms(df, cond_onset_type=''):
    plt.figure()
    ax = df['condition_onset_minus_fitbit_start_days'].hist(bins=20, edgecolor='c', linewidth=2)
    if cond_onset_type=='Incident':
        _ = plt.xticks(ticks=45+np.arange(0,2550,126), rotation=90)
        _ = plt.xlim([0,2625])
 
    _ = plt.title(cond_onset_type+f' Meta Cases, total={len(df)}')
    _ = plt.xlabel('Days between first valid fitbit day and condition')
    ax.xaxis.label.set_fontsize(12)
    
    plt.figure()
    ax = df['num_occurences'].hist(bins=20, facecolor='purple', edgecolor='m', linewidth=2)
    _ = plt.xticks(ticks=np.arange(0,2550,121), rotation=90)
    _ = plt.xlabel('Number of valid days with fitbit info')
    ax.xaxis.label.set_fontsize(16)
    
    plt.figure()
    all_condition_names = []
    for cond in df['standard_concept_name']:
        all_condition_names.extend(cond)

    condition_names_unq, condition_names_counts = np.unique(all_condition_names, return_counts=True)

    cond_count_sort_inds = np.argsort(condition_names_counts)

    plt.barh(np.arange(0,len(condition_names_unq)), [condition_names_counts[i] for i in cond_count_sort_inds], edgecolor='c')
    _ = plt.yticks(ticks=np.arange(0,len(condition_names_unq)), labels=[condition_names_unq[i] for i in cond_count_sort_inds])
    _ = plt.title('Condition Names for '+cond_onset_type+ f' Cases ({sum(condition_names_counts)} conds. for {len(df)} subjects)')
    plt.grid(which='both')
    
    plt.figure()
    all_condition_names = []
    for cond in df['standard_concept_name']:
        all_condition_names.append(cond[0])

    condition_names_unq, condition_names_counts = np.unique(all_condition_names, return_counts=True)

    cond_count_sort_inds = np.argsort(condition_names_counts)

    plt.barh(np.arange(0,len(condition_names_unq)), [condition_names_counts[i] for i in cond_count_sort_inds], edgecolor='c')
    _ = plt.yticks(ticks=np.arange(0,len(condition_names_unq)), labels=[condition_names_unq[i] for i in cond_count_sort_inds])
    _ = plt.title('Condition Names for '+cond_onset_type+ f'Cases ({sum(condition_names_counts)} conds. for {len(df)} subjects)')
    plt.grid(which='both')

In [ ]:
incident_cases_df = fitbit_condition_df.loc[fitbit_condition_df['condition_onset_minus_fitbit_start_days']>0]
create_data_histograms(incident_cases_df, cond_onset_type='Incident')

In [ ]:
prevalent_cases_df = fitbit_condition_df.loc[fitbit_condition_df['condition_onset_minus_fitbit_start_days']<0]
create_data_histograms(prevalent_cases_df, cond_onset_type='Prevalent')

In [ ]:
dataset_13847333_age_sql = """
    SELECT
        * 
    FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_ehr_data = 1 """

both_df_all_meta = both_df_all_meta.drop(columns = ['dob'])
both_df_all_meta = both_df_all_meta.merge(df_person[['person_id','dob']],on ='person_id')

df_age = pd.read_gbq(
    dataset_13847333_age_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

both_df_all_meta = both_df_all_meta.merge(df_age[['age_at_cdr','age_at_consent','person_id']],on  = 'person_id')
both_df_all_meta['age_date'] = (
    pd.to_datetime(both_df_all_meta['dob']) +
    pd.to_timedelta(both_df_all_meta['age_at_consent'] * 365.25, unit='D')
)
both_df_all_meta['condition_start_datetime'] = both_df_all_meta['condition_start_datetime'].apply(lambda x: str(x).split(' ')[0])
both_df_all_meta['condition_start_datetime'] = pd.to_datetime(both_df_all_meta['condition_start_datetime'])

WINDOW = pd.Timedelta(days=30)
prevalent_all_meta = both_df_all_meta[
    both_df_all_meta['condition_start_datetime'] <=
    (both_df_all_meta['age_date'] + WINDOW)
]
incident_all_meta = both_df_all_meta[
    (both_df_all_meta['condition_start_datetime'] >
     (both_df_all_meta['age_date'] - WINDOW)) &
    (~both_df_all_meta['person_id'].isin(prevalent_all_meta['person_id']))
]
len(set(prevalent_all_meta['person_id'])),\
len(set(incident_all_meta['person_id'])),\
len(set(prevalent_all_meta['person_id']))+\
len(set(incident_all_meta['person_id']))

In [ ]:
both_df_all_meta.to_csv('meta_cases_031826.csv',index = False)
controls_meta.to_csv('meta_controls_031826.csv',index = False)

In [ ]:
both_df_all_meta_qc_f.to_csv('fitbit_meta_cases_031826.csv',index = False)
control_all_meta_qc_f.to_csv('fitbit_meta_controls_031826.csv',index = False)
prevalent_cases_df.to_csv('fitbit_meta_prevalent_cases_031826.csv',index = False)
incident_cases_df.to_csv('fitbit_meta_incident_cases_031826.csv',index = False)

In [ ]:
#new line
both_df_all_meta = pd.read_csv('meta_cases_031826.csv')
controls_meta = pd.read_csv('meta_controls_031826.csv')
both_df_all_meta_qc_f = pd.read_csv('fitbit_meta_cases_031826.csv')
control_all_meta_qc_f = pd.read_csv('fitbit_meta_controls_031826.csv')
prevalent_cases_df = pd.read_csv('fitbit_meta_prevalent_cases_031826.csv')
incident_cases_df = pd.read_csv('fitbit_meta_incident_cases_031826.csv')
both_df_all_meta['age_date_1'] = (
    pd.to_datetime(both_df_all_meta['dob']) + pd.to_timedelta(1 * 365.24, unit='D') +
    pd.to_timedelta(both_df_all_meta['age_at_consent'] * 365.25, unit='D')
)
both_df_all_meta['age_date_2'] = (
    pd.to_datetime(both_df_all_meta['dob']) + pd.to_timedelta(2 * 365.24, unit='D') +
    pd.to_timedelta(both_df_all_meta['age_at_consent'] * 365.25, unit='D')
)
both_df_all_meta['age_date'] = (
    pd.to_datetime(both_df_all_meta['dob']) +
    pd.to_timedelta(both_df_all_meta['age_at_consent'] * 365.25, unit='D')
)

WINDOW = pd.Timedelta(days=30)
prevalent_all_meta = both_df_all_meta[
    both_df_all_meta['condition_start_datetime'] <=
    (both_df_all_meta['age_date'] + WINDOW)
]
incident_all_meta = both_df_all_meta[
    (both_df_all_meta['condition_start_datetime'] >
     (both_df_all_meta['age_date'] - WINDOW)) &
    (~both_df_all_meta['person_id'].isin(prevalent_all_meta['person_id']))
]
prevalent_all_meta2 = both_df_all_meta[
    both_df_all_meta['condition_start_datetime'] <=
    (both_df_all_meta['age_date_2'] + WINDOW)
]
incident_all_meta2 = both_df_all_meta[
    (both_df_all_meta['condition_start_datetime'] >
     (both_df_all_meta['age_date_2'] - WINDOW)) &
    (~both_df_all_meta['person_id'].isin(prevalent_all_meta2['person_id']))
]
prevalent_all_meta1 = both_df_all_meta[
    both_df_all_meta['condition_start_datetime'] <=
    (both_df_all_meta['age_date_1'] + WINDOW)
]
incident_all_meta1 = both_df_all_meta[
    (both_df_all_meta['condition_start_datetime'] >
     (both_df_all_meta['age_date_1'] - WINDOW)) &
    (~both_df_all_meta['person_id'].isin(prevalent_all_meta1['person_id']))
]
import numpy as np
import matplotlib.pyplot as plt

prev = [
    len(set(prevalent_all_meta['person_id'])),
    len(set(prevalent_all_meta1['person_id'])),
    len(set(prevalent_all_meta2['person_id']))
]

inc = [
    len(set(incident_all_meta['person_id'])),
    len(set(incident_all_meta1['person_id'])),
    len(set(incident_all_meta2['person_id']))
]

x = np.arange(3)

plt.figure()

bars1 = plt.bar(x - 0.2, prev, width=0.4, label="Prevalent")
bars2 = plt.bar(x + 0.2, inc, width=0.4, label="Incident")

# ---- Add annotations ----
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2,
            height,
            f'{int(height)}',
            ha='center',
            va='bottom'
        )

plt.xticks(x, ["case ≤ \nstart_fu+30D", "case ≤ \nstart_fu + 1y +30D", "case ≤ \nstart_fu + 2y +30D"])
# plt.xlabel("Dataset Version")
plt.ylabel("No of individuals")
plt.title("Prevalent vs Incident Comparison\n age_at_consent (start_fu)")
plt.legend()
plt.grid(axis='y')

plt.show()


In [ ]:
len(set(both_df_all_meta['person_id']))

In [ ]:
prevalent_meta_df_p = prevalent_all_meta2.groupby('person_id').agg({'condition_start_datetime':min,'age_date_2':'first'}).reset_index()
incident_meta_df_p = incident_all_meta2.groupby('person_id').agg({'condition_start_datetime':min,'age_date_2':'first'}).reset_index()
incident_meta_df_p['meta_label'] = 2
prevalent_meta_df_p['meta_label'] = 1
meta_df_p = pd.concat([prevalent_meta_df_p,incident_meta_df_p],ignore_index = True)
meta_df_p['date_consider_old'] = meta_df_p['condition_start_datetime']
meta_df_p['date_consider'] = meta_df_p['age_date_2']

In [ ]:
# controls_meta = controls_meta.merge(df_person[['person_id','dob']],on ='person_id')
# controls_meta = controls_meta.merge(df_age[['age_at_cdr','age_at_consent','person_id']],on  = 'person_id')
controls_meta['age_date'] = (
    pd.to_datetime(controls_meta['dob']) +
    pd.to_timedelta(controls_meta['age_at_consent'] * 365.25, unit='D')
)
controls_meta['age_date_1'] = (
    pd.to_datetime(controls_meta['dob']) + pd.to_timedelta(1 * 365.24, unit='D') +
    pd.to_timedelta(controls_meta['age_at_consent'] * 365.25, unit='D')
)
controls_meta['age_date_2'] = (
    pd.to_datetime(controls_meta['dob']) + pd.to_timedelta(2 * 365.24, unit='D') +
    pd.to_timedelta(controls_meta['age_at_consent'] * 365.25, unit='D')
)
controls_meta['meta_label'] = 0
controls_meta['date_consider'] = controls_meta['age_date_2']

In [ ]:
cohort_meta_df = pd.concat([controls_meta,meta_df_p],ignore_index = True)[['person_id','date_consider','meta_label']]
cohort_meta_df

In [ ]:
control_all_meta_qc_f = control_all_meta_qc_f.merge(df_person[['person_id','dob']],on ='person_id')
control_all_meta_qc_f = controls_meta.merge(fitbit_hr_df[['person_id','first_date']],on  = 'person_id')
control_all_meta_qc_f['meta_label'] = 0
control_all_meta_qc_f['date_consider'] = control_all_meta_qc_f['first_date']

In [ ]:
incident_cases_df['date_consider_old'] = incident_cases_df['condition_start_datetime_earliest']
incident_cases_df['date_consider'] = incident_cases_df['first_date']
incident_cases_df['meta_label'] = 2
prevalent_cases_df['date_consider_old'] = prevalent_cases_df['condition_start_datetime_earliest']
prevalent_cases_df['date_consider'] = prevalent_cases_df['first_date']
prevalent_cases_df['meta_label'] = 1

In [ ]:
cohort_fitbit_meta_df = pd.concat([control_all_meta_qc_f,incident_cases_df,prevalent_cases_df],ignore_index = True)[['person_id','date_consider','meta_label']]
cohort_fitbit_meta_df

### Charleston fucntion based

In [ ]:
import pandas as pd
import os

#Peptic ulcer disease
PUD_SNOMED_IDS = [4027663]
PUD_ICD_IDS = [45609985,44820383,45938535]

#Peripheral vascular disease
PVD_SNOMED_IDS = [321052,44782775,3654996,4131908,4325344,44782426,4124836,36713094,760977]
PVD_ICD_IDS = [1569324,35207860,44826654,44835959,1569326,45567223,44834745,44819724,3353610]

#Chronic pulmonary disease
CPD_SNOMED_IDS = [255573,257004,4110056,4195892,252348,46274062,4196712,4193588,4209097,44782563,4046986]
CPD_ICD_IDS = [1569487,35208025,35208024,35208023,44824239,44827786,44828976]

#Cerebrovascular disease
CVD_SNOMED_IDS = [381591,434056,42539256,40481762,40480002,4121341,43530687,43531622,43531621,40481842,43531583,\
40480938,40484513,40484522,40479575,40480449,43530702,605223,40484120,\
4046444,43530688,43530679,40480475,44782427]
CVD_ICD_IDS = [1569225,1569228,35207836,44832388,44831252,44835952,35207837,44821960,45576890,1569264,\
44827802,44828989,44832390,44837112,44828990,1569268,44825445,1569257,1568361,\
44832389,1569270,44834740,44826650,44833578,45562384,45567205,44834741,45596225,\
1569261,45548053,44835955,45538418,44819717,1569263,44826648,1569265,44826649,\
1569227,44835956,44824256,45552822,44828986,37200555,45591492,45538419,44830092,\
45562381,45548051,44835954,45567199,45567206,1569258,44820876,37200561,44823122,\
45567203,37200547,44828987,44828988,45581797,45562383,1595599,45572115,1569266,\
45576900,35207408,45543203,1569267,45605823,44823123,37200554,45538416,44837113,\
45557565,44821959,44832391,45562382,45543204,45601063,45543208,37200553,45576901,\
45567202,44831254,45596226,37200559,37200558,44827803,1569260,44837114,44837115,\
1569259,35207840,45567200,44831253,45548050,45557567,45572114,44824255,45548055,\
45557566,37200560,37200550,45601064,1595601,37200552,37200546,37200551,45552818,\
45596224,44820877,45552819,45567204,1569262,1569269,45567201,44824254,44831255,\
45538420,45591491,45572113,45552820,45533481,45586612,37200548,37200556,45601062,\
45562380,45581795,45543207,45605825,45605827,37200549,37200557]

#Hemiplegia or paraplegia
HEMI_SNOMED_IDS = [343531639,374022,44782781,40480429,40482237,40481762,375528,40484513,\
40480944,40480435,40484522,372613,40481820,42535482,40481757,42535481,\
40484910,134031,374336,4083063,42539286,42535480,4175708,42535477,\
42535483,42535406,42535404,42535405,192606,43530719,43530718,4101733,\
192901,4101739,37019108,44784249,44784250,4309219,42537693]
HEMI_ICD_IDS = [1568408,1568411,1569254,45533479,45571830,44832275,45538140,45591489,\
44834632,44829966,44832390,1569268,44825445,45572111,45542906,1568410,\
45567205,1569261,45538418,44826648,44825334,45557257,45566902,44828986,\
44824157,45586315,45591492,45538419,1569240,45543201,1568409,45548051,\
44828871,44833441,44834631,44827695,44834630,1569247,45571829,44819584,\
45543198,45543202,45562105,45538405,44828869,44828873,45538138,44826552,\
45567197,45600806,45538408,45600807,1569233,45576900,44828868,45605573,\
44823019,44833442,45543208,45605814,45538139,45576895,45567200,44828870,\
45562373,44824158,45601064,45596224,45576896,45571828,45572105,45596221,\
45543196,45576609,45538409,1568412,1568413,45581515,44824159,45586316,\
45566903,35207319,44832266,35207306]

#renal disease
RENAL_SNOMED_IDS = [198124,46271022,201313,192279,194686,35624383,43531578,195556,439696,4177206,\
193782,193493,44784439,199870,197921,4128220,45757752,192679,442766,4024556,\
37018886,43020455,4122924,442626,439695,439694,442603,4128228,43531653,43531577,\
45768813,42537890,4128224,764011,760850,43531559,43531566,43531562,46273164,45769906,\
4153346,4135463,4194065,45768672]
RENAL_ICD_IDS = [45548653,44830077,35207671,44827889,35207674,44828971,44825427,1571484,44826634,\
45596188,44831231,35209268, 35209267,44825426, 44834717, 44824235, 44821950, \
44820856,35209266,44819693,3372285, 44833560, 44831232]

#aids
AIDS_SNOMED_IDS = [4228133,4228440,4228277,4225643,4226905,4225318,4226110, 4128060, 4221489, 4267414]
AIDS_ICD_IDS = []

#leukemia
LEUK_SNOMED_IDS = [317510,132853,140057,138379,138708,140666,4082311,135766,140352,\
135496,134596,134305,134597,134603,132850,133438,135762,141816,\
132572,4038845,4133599,4097706,36715587,4002497,4003188,133154,\
321526,4189938,4079282,4137687,40483761,443719,135768,315497,\
136056,133158,36712834,4001331,135499,608277,40481524,4173974,\
4079686,760936,313159,3572249,4003187,4079683,4173963,44501329,\
4082460,4002496,4082459,760933,136656,313430,4143382,36717461,\
140672,4230253,3654651,4211481,4091925,760962,760935,36676614,\
4301780,760934,4082461,4173955,44783718]
LEUK_ICD_IDS = [1567680,1567682,45571568,1567689,44821758,35225309,35225341,\
44825232,1567690,44831013,44834519,1567711,45547554,1567688,\
45595719,1567714,44820645,45590968,1567681,44829849,1567691,\
44826430,45576368,45581255,45557039,44827588,44819465,45561871,\
44832156,44829852,44821759,44833065,1567705,45557040,44828757,\
44829844,44822655,1567712,45595721,45532959,45571579,1567698,\
44821760,44825231,1567687,44821501,1567699,45561879,45595726,\
45595720,1567701,1567697,44819470,45557046,44832157,1567694,\
45581253,45532961,44835715,1567684,1567713,44819472,44820642,\
44835713,1567685,45537877,44835414,44828758,45590964,44820643,\
45542669,45566647,45561873,44826432,1567695,45542662,1567677,\
44819471,44831017,44827586,1567710,45542665,45547557,1567692,\
44824053,45557038,45547552,44821761,44831015,45590967,44828760,\
45605332,1567700,45557043,44830740,45542668,45557045,45581257,\
45561875,44819469,44836875,45547555,44821756,45566649,44824054,\
44829585,44833339,44831019,45595723,44822909,45552313,45586065,\
45576366,44822907,45576369,45561872,44834520,1567706,1567696,\
45586066,44826433,45566651,45581261,44822910,44825230,44820646,\
45561877,44834521,44827584,45542664,44821757,1567683,45590966,\
45605331,45571573,44829851,45595716,45552317,1567707,45571574,\
44504688,44835716,45576371,44826436,44829847,45586063,45595717,\
45586068,44833342,44835714,45586069,44821762,45586070,45532963,\
45600567,45532958,45571569,1567686,45537874,44831016,44832163,\
45571577,45532965,44832161,44501329,1567708,44832165,44820641,\
44822912,44833344,45571578,45537878,45586067,45561876,44833345,\
44824055,45566648,44820647,44832155,44829850,44832158,44832160,\
44504701,44501122,44504700,44833336,45605335,45532964,45600570,\
45547553,45537875,1567704,45600569,44834522,44833338,44829848,\
1567703,45605334,44825234,44821763,44833343,45586072,44821502,\
45552311,45595718,45586064,44822908,44832162,44825233,44835718,44832159]

#myocardial infarction
MI_SNOMED_IDS = [4329847,314666,312327,4270024,4296653,37309626,765132,434376,\
438170,43020460,44782712,46270163, 438438,319038,4108217,\
4108680,441579,45766241,438447,436706,4121477,44782769,46270162,\
45766114,439693,4108218,4108677,438172,4124687,4119953,4108679,\
4200113,4322145,4064609,4121467,45766116,4108220,45766075,3654466,\
4243372,4303359,4064348,4108678,4170094,4119949,4108219]
MI_ICD_IDS = [35207702,1569126,35207685,44832372,44820864,1326589,1326590,\
44819702,44835928,44819701,35207684,1326588,1569128,1569127,\
45605779,44834718,45576865,1569129,44825428,45572079,45557536,\
44825430,45533436,44819697,35207698,44831236,1569131,44820857,\
44835926,44834719,44834723,44834733,44820861,44824237,1569130,\
44827782,35207696,44832373,44833561,44826643,44831238,44826636,\
1326591,45605781,44819698,45572080,44834724,44830079,44823111,\
44819700,44832374,35207697,45562340,44832375,44819699,44834720,\
44834722,44832376,35207687,35207689,35207686,44826635,35207691,\
44834721,44820858,35207688,44820859,44820860,35207693,35207692,35207695,35207694]

#liver disease
LIVER_SNOMED_IDS = [194984,4212540,194990,4341650,201613,201612,37017009,4340391,\
193256,4055224,201343,40481531,444421,197654,4026131,4059297,\
4026136,4058694,4058695,4219504,4277276,45769564,46273476,4059298,\
4068119,4059299,194280,36716710]
LIVER_ICD_IDS = [1569684,44826726,44836029,35208368,35208367,45605957,1569683,\
35208357,1569669,35208358,44830168,35208332,1569673,44823185,\
44834823,35210961,35208340,45543341,35208337,35208369,44824570,\
1569674,45557672,35208333,35208338,35208334,35208339,1569675,\
45543335,35208335,45586731,45552948]

#Lymphoma
LYM_SNOMED_IDS = [432571,4038838,434592,135215,4131931,4173973,4079691,138379,4003830,440058,\
4038835,432574,4300704,132841,438698,4003183,4147411,200349,4149840,40487528,\
4082487,40491472,40489940,4266193,4266805,435753,4002357,40490918,4079692,\
4082488,42538850,4002356,441521,4003833,40482893,320347,192560,4141255,\
40481901,4001329,200343,40490458,195195,36684807,36684814,4038839,434881,\
194878,4041800,4038841,200338,36684809,45765770,435207,4123298,373152,\
40481357,4119130,4003188,4001328,441235,4029188,4294434,4038843,40481522,\
40486171,4142582,36684461,40488896,315481,440350,40479608,40490465,4294435,\
198088,435492,4301668,4184976,4097560,4299149,4002494,198707,4116341,4212994,\
4079288,437505,35610328,4144302,36684463,4173976,36684812,35622279,198374,\
44808122,4161665,4301666,4091768,4082627,4079696,4287493,4307552,193428,\
606919,35610325,40483374,196055,439293,40493027,4003836,4150201,4080742,\
36525655,4174765,4308266,4301661,4226624,37118088,4045412,36684464,439267,\
36684813,42538056,4299150,3654651,36528823,4097561,44501047,36567132,44499593,\
44499591,4121330,4096969,4003831,36521653,4079289,40490467,40492018,40489407,\
36550153,36553646,36526964,4082491,4174442,40493029,4344254,4301658,4301659,\
4300783,4094544,4080741,36536260,760724,36684462,40480206,603291,40489468,40491440,4297356,35622133]
LYM_ICD_IDS = [1567668,1567672,45590959,44829843,44835708,1567651,45609981,\
1567654,1567669,1567671,45561851,1567641,1567633,45590954,1567650,\
1567640,1567674,45600540,45561866,45552300,1567652,45561658,44829838,\
1567659,45557027,44833325,45566645,45600547,45590960,45571554,1567642,\
920123,44835709,44834505,1567665,45561852,44831011,1567643,45557023,45590951,\
1567653,45542647,1567649,44825218,1567635,1567639,45576357,45542658,44820628,\
45557036,1567634,44832154,45552302,1567657,45547538,45561849,45581247,1567673,\
1567662,45600559,45576365,44836862,45581228,44821750,45595705,45581229,44833313,\
45590932,44822893,45581233,45542645,45595704,44829826,45590938,45542652,45557035,\
45576364,44832137,1567644,45552297,45581240,1567645,45537847,44836870,45590943,\
44828755,45561842,44822900,45547548,45532945,45532951,1567656,45571546,45566643,\
44835693,45537859,1567685,44820627,45537852,45581230,45542637,1567655,44824052,\
44822906,45576348,45590955,45566635,45566634,1567670,45561873,45561839,44833330,\
45532944,45561853,45571550,45586045,45561850,45586039,45547549,45581246,45547540,\
45590941,45561864,45537851,45561844,45581239,45547550,44821748,1567658,45561868,\
1567647,45557024,45581234,1567636,45605317,45561860,45576350,1567646,45590956,\
45605298,1567664,44826418,45600541,1567638,45586040,44830994,45576351,44822895,\
45605326,44828750,1567648,45586043,45566633,1567667,45571557,45561841,45566642,\
45537869,45595713,45561843,45600539,45547544,45605310,45595694,45586031,45552294,\
45542655,44833314,920121,45532952,1567663,44835703,44835710,45537868,1567654,45566630,\
45532953,45537870,44819463,45566644,45605304,45561854,45566628,45605305,45566629,\
45586034,45571547,45586041,45586057,45595703,45552313,45547537,45571551,45557019,\
45600558,44822894,45561847,45566639,45532955,45537867,45537850,45590942,44836856,\
45552308,45566621,45600560,1567666,45532950,45542653,45605327,45557034,45576358,\
44832138,45557025,45557020,45590929,45552290,44832146,44827578,45566636,44835707,\
45600550,45552301,45571548,44830996,45561865,44826429,45600555,45605301,45557026,\
45552298,45552296,44836857,45571565,920116,45566630,45557022,44821749,45566627,\
45552304,45586037,45595710,44836855,45590933,45595700,45595702,45542640,45581238,1567637,44834506,45542634]


#Malignancy
MALIG_SNOMED_IDS = [4147164,198700,4181328,46271027,4233629,438701,4201930,4264693,\
45770837,4264112,4094540,4299320,4096968,43531175,42536908,45757385,\
45757386,4166232,4094541,4094542,37312593,764422,36716177]
MALIG_ICD_IDS = [44831899,44833044,44831898,44834240,44835411,44830737,\
44835413,44833045,44828488,44831896,44820372,44837751,\
44837750,44835416,37202274,44821498,44833043,44835410,\
44834216,44833042,37202275,37202276,44837753,45548989,44828169,\
44835101,44830738,45544160,45606828,44836599,44820371,44829584,44827307]

#congestive heart failure
CHF_SNOMED_IDS = [319835,439696,319034,4229440,314378,313502,4242669,\
4023479,44782655,44782728,439698,314369,4284562,37309625,\
4139864,44784442,315295,439694,43022068,316994,43021826,44782713,\
36713488,36712927,4215446,4206009,4142561,36712928,4236658]
CHF_ICD_IDS = [1569180,1569179,45533456,45576878,44826642,45601038,45586587,1569181,\
45562355,45567180,45586588,45543182,45567181,45548022,45591469,45533457,44823108]

#diabetes
DM_SNOMED_IDS = [40480031,35626069,45769905,40480000,4099216,45757266,4147719,609102,\
434480,45757507,37312204,609098,443592,609109,4245270,4143857,40482883,\
43531011,35626763,4215719,37312207,43531577,4143529,4177050,37018728,\
45757362,4016047,43531640,4164730,4060085,45757432,609112,602345,4144583,\
37016356,45769873,37311832,609097,36712686,609107,609100,44808385,45757499,\
4178452,45757674,609104,609119,4230254,43531651,609108,37311833,609105,\
36712687,609115,609118,609106,609117,609120,609103,761048,37016357,\
37017430,4206115,4224254,4030664,42535540,45757508,43531653,4228112,\
42537681,4099334,4228443,201531,4128221,35626044,45757449,35626088,\
4171406,35626068,35626043,35626087,35626067,37016358,3655382,609116,3655383,\
761051,45769832,4304377,4225656,37016180,4226798,4191611,4189418,\
4099214,4152858,37016355,761049,4200875,4338901,4095288,43531564,\
36674200,4234742,36714116,36674199,4140466,45769830,45773064,35626041,\
45770880,4223303,4145827,438476,609096,35626042,40484649,45771064,609095,\
4227657,4225055,609114,45773688,609099,609101,35626039,201530,35626038,\
4130162,4131908,37016767,318712,4029420,376114,45763583,30968,45763584,\
36715571,439770,37016179,45770902,37311254,45757474,37311253,4266637,\
35626764,4290822,4063042,4114427,40485020,43531563,36717156,443734,\
4222415,4063043,37017431,4202383,4048028,4226238, 4009303,37110593,\
4033942,43531616,4334884,35626904,380097,377821,380096,40484648,\
200687,42538169,43530685,377552,4099651,35626070,45770928,45770830,\
37312218,4227210,45770881,443735,45769876,4226354,4222876,37018566,4196141,\
376979,4221495,4175440,443727,42536605,37016768,40482801,37016348,\
201820,201826,4008576,4193704,442793,443732,4034964,37311673,37016349,\
443730,376065,4082346,192279,4044391,443731,4311708,376112,37016354,\
43530689,43531578,37017432,443767,443238,443733,321822,443729,201254,\
4174977,443412,376683,4029423,4226121,195771,435216,4311629,45757363,\
43530656,378743,4159742,45757435,45757277,43530690]
DM_ICD_IDS = [1567956,35206882,44833365,44834548,44836914,1567967,45605405,1567965,\
1567958,35225076, 45595797,45605403,45557113,45595799,1567959,35206881,\
44836915,1567966,44827615,44828795,1567940,1567972,45591027,44819503,\
1567971,1567906,45591031,1567969,44825349,45533021,44828794,45576443,\
44831047,44820682,44824074,35206879,44819500,1567960,1567961,45581355,\
45605401,45581354, 44831045,44820685,45581353,45595798,1567951, \
35206885,45605404,44821617,45537962,45533018, 44827617,45547627,1326491,1326492,44832187,\
45533019,1567949,37200204,1567964,1567957,44829880,37200205,45542738,\
44833366,1567942,1567943,45600642,44821787,1567918,44827616,1567962,\
45552386,1567955,1567909,45600640,1567916,1567984,44828793,45533022,\
45547621,44832194,1567941,45600636,45600638,1567968,1567923,45591029,\
45595795,1567974,45566733,44829882,45552372,45581352,44829879,35206878,\
45600641,44826461,1567982,45591030,44836911,45552385,45547624,45547625,\
44820680,44831046,1567944,1567910,1567907,37200203,37200202,44822936,37200248,\
45571652,45542736,45586132,1567920,45537961,45581344,45533015,44826459,\
45552379,45595792,45533017,37200224,44836918,44829878,37200212,1567953,\
1567963,37200249,1567950,44824072,45547617,44822935,45605392,45595805,1567948,\
37200200,45605398,44829881,1567945, 44822934,44833368,37200208,37200241,44836916,\
1567975,45591026,45547622,35206911,1567976,44825262,1567935,\
44821786,44829876,44828790,45600637,45571654,45557112,1567986,\
45533009,45561956,45605402,44833367,45557109,45605395,45566735,\
37200247,44822938,35206884,44819501,45566736,44828788,37200244,\
44822932,37200246,45547626,1567912,45600644,45576438,44820684,45586138,\
37200222,45561955,44834549,37200223,37200210,1567915,45586140,\
44824073,45586139,37200198,44824071,44834547,37200211,37200147,\
44836917,45600634,37200207,37200206,37200216,37200213,1567977,\
44833364,37200191,45576437,37200225,37200199,1567908,45581349,44819504,\
37200148,1567946,37200201,37200220,45552382,45595789]


#rhematogic
RHEUM_SNOMED_IDS = [4179092,75621,4295179,4270868,437082,4098292,45773068,36716891,37209321,37209323,\
          37209322,43530808,4005037,72714,4291433,4324123,4025957,80182,4081250,4344161,\
          432919,606386,606388,4063581,46273369,81097,4114444,37016279,4266078,\
          42534954,74125,606427,4083681,37395588,4259507,4253901,4132810,\
          4132809,72401,4103749,4193620,4103019,4055369,4055640,\
          255304,4009504,75622,4107913,4105026,46270384,37399735,\
          37017494,72705,4101469,4265764,31384009,\
          4346977,46270482,4102493,4173925,4347058,73854,40485046,\
          40319772,46274123,4083682,4064048,4319305,78357,606430,\
          4145240,4306357,4269880,80809,4117687,4115161,4116445,\
          36687005,36687002,36687006,4116439,4115050,4116440,4114441,\
          4179378,4116150,36683391,4116151,36685017,36685018,36685019,\
          42534834,42534835,37108590,35609009,36685020,4117686,36685021,\
          36685022,36685023,42534836,42534837,37108591,35609010,36685024,\
          4114439,4116446,4116441,4103516,36684997,36684998,256197,4030424,4271003,\
          37311514,4295306,37117740,4344163,4083556,37207809,4035611,37108714,\
          254443,4285717,4250483,4297164,4267801,4178133,4002526,4066952,4343923,\
          257628,4318863,4344158,4149913,4219859,4116447,607145,134442,\
          4063582,4185187,4027727,46270481,4218311]
RHEUM_ICD_IDS = [44835004,1570654,45951248,44833839,45548419,45533841,45586969,45601454,45548418,45533840,\
45553184,45543577,45582150,45567561,45552361,1569908,44828023,44822181,44826855,44819943,\
44821103,44836160,44826856,44834965,44834964,44831477,44833806,44834963,44827743,45566915,\
45547777,45562120,35208825,45601433,45562709,35208754,44837332,1570615,45591821,45533827,\
45557864,45567547,1570090,1570097,1570092,45601299,45548279,45591709,45596448,1570091,\
45553058,35208755,45567545,44837143,44821124,766401,766404,766408,766409,766411,766410,\
766406,766402,1570616,45591822,45553168,45572446,1326705,45562710,35208826,1570620,45606214,\
1570039,1570031,1570038,1570033,1570035,1570036,1570037,45586839,45601281,45582016,45601283,\
45553048,45533698,45596440,45553049,725371,45548265,45601280,45586838,45543436,45562592,\
45572339,1570032,45533699,45601282,45533700,45548264,45557761,45553046,45553047,1570034,\
44836170,1570064,1570071,1570066,1570068,1570069,1570070,45591705,45562598,45562599,45543443,\
45562600,45567422,45567423,45586845,725373,45606064,45538639,45606063,45543442,45601289,\
45572346,45577117,1570065,45567425,45596446,45577119,45567424,45548271,45601288,45548270,\
45577118,45567426,1570067,1569900,45572311,45572448,44834974,1570098,1570105,1570102,\
1570103,1570104,45533707,45586854,45548280,4554828,45591711,45557772,45577126,45577127,\
45562603,1570099,45601302,45586853,1570101,45601434,1569892,45562569,35208758,44837333,\
44834973,44831474,1570617,45557865,45591823,45596564,45548402,35208828,45548197,\
44820580,1569950,1569957,1569952,1569954,1569955,1569956,45533684,45557750,45572322,\
45553031,45601271,45567400,45606037,45572321,45586828,45596421,45601270,45596420,1569951,\
45582000,45586827,45548246,45586829,45577089,45548244,45548245,45553032,1569953,44831487,\
44825676,1570022,1570019,1570020,1570021,45606049,45572335,45533696,45591699,45577101,\
45567417,45553043,45591698,45548260,45606047,45596435,45577102,1570016,45601279,45572336,\
45567416,45606048,45548259,1570018,1570015,1569966,1570030,1570025,1570027,1570028,1570029,\
45591700,45572337,45596437,45538633,45548263,45543435,45582014,45553045,725370,45596438,\
45548261,45606051,45572338,45548262,45596436,45606050,1570024,45596439,45562591,45582015,\
45567419,45533697,45567418,45543434,45553044,1570026,1570023,35208750,1570040,1570047,1570042,1570044,\
1570045,1570046,45567420,45577104,45572340,45533702,45553051,45533701,45562593,45572341,725372,\
45577109,45557762,45606055,45557763,45601284,45606053,45606054,1570041,45533703,45577105,\
45577107,45577108,45538635,45591701,45596442,45577106,45606056,1570043,35208753,1569991,\
45567409,1569992,45586836,1569994,1569975,1569982,1569979,1569980,1569981,45548254,4555303,\
45557754,45601274,45591688,45577095,45577094,45591686,1569999,1570003,45577100,45572332,\
1570007,1570014,1570011,45582013,45538631,45567415,45538632,45538629,1570008,45543433,45567413,\
1569983,1569990,1569987,45596428,45538624,45572327,44829682,44828021,1570622,766396,766400,\
766397,766398,766394,45533829,45577246,45577247,45557867,766395,45557866,766399,45548403,1569884,45533671,\
35208700,44819941,1570612,1570613,45582126,35208827,44832613,1570619,35208830,45596565,45553171,\
45582130,35208831,45591820,1570082,1570084,45567431,1570089,1570086,1570087,1570088,45567434,\
45562602,45606069,45572348,45543446,45548275,45606071,45543445,45606068,45538643,45606070,\
45567432,45601293,1570083,45586849,45601295,45538644,45553057,45557769,1570085,44832637,\
44830328,44819953,44836178,44834984,44829202,44819954,44833819,44829201,44822195,44834983]



DEM_SNOMED_IDS = [35608576, 37018688, 37109056, 37110513, 374888, 375791, 376085, 376095, 376946, 377254, \
377527, 378125, 378419, 378726, 379778, 379784, 380701, 380986, 381832, 4009647, 4043378, \
4046090, 4047745, 4047747, 40483103, 4048875, 4092747, 4101137, 4139421, 4182210, 4196433, \
4218017, 4220313, 42538857, 4277444, 4278830, 43021816, 43530664, 43530666, 441002, 443605, \
443790, 443864, 444091, 44782763, 44782771, 762497]
DEM_ICD_IDS = [1568088, 1568090, 1568295, 35207114, 35207337, 35207356, 35207357, 35207358, 35207359, 35207360,\
37402371, 37402372, 37402373, 37402374, 37402375, 37402377, 37402379, 37402380, 37402383,\
37402385, 37402388, 37402390, 37402393, 37402395, 37402398, 37402399, 37402400, 37402401, \
37402402, 37402403, 37402404, 37402406, 37402409, 37402410, 37402411, 37402412, 37402414, \
37402416, 37402417, 37402418, 37402419, 37402420, 37402422, 37402424, 37402425, 37402426, \
37402427, 37402429, 37402430, 37402431, 37402432, 37402433, 37402435, 37402437, 37402438, \
37402439, 37402440, 37402441, 37402443, 37402445, 37402446, 37402447, 37402448, 37402451, \
37402453, 37402454, 37402456, 44819534, 44820708, 44820709, 44821810, 44821811, 44821814, \
44824106, 44824152, 44826537, 44826538, 44827641, 44827644, 44829914, 44829915, 44829917, \
44831078, 44831079, 44831083, 44831122, 44832219, 44833435, 44834581, 44835772, 44835773, \
44835825, 44836954, 44836959, 45538000, 45538103, 45547730, 45591073, 45591076, 45595842, \
45595843, 45600684, 45605533]

In [ ]:
COMORBIDITIES = {
    "myocardial_infarction": {
        "snomed": MI_SNOMED_IDS,
        "icd": MI_ICD_IDS,
        "points": 1
    },
    "congestive_heart_failure": {
        "snomed": CHF_SNOMED_IDS,
        "icd": CHF_ICD_IDS,
        "points": 1
    },
    "peripheral_vascular_disease": {
        "snomed": PVD_SNOMED_IDS,
        "icd": PVD_ICD_IDS,
        "points": 1
    },
    "cerebrovascular_disease": {
        "snomed": CVD_SNOMED_IDS,
        "icd": CVD_ICD_IDS,
        "points": 1
    },
    "chronic_pulmonary_disease": {
        "snomed": CPD_SNOMED_IDS,
        "icd": CPD_ICD_IDS,
        "points": 1
    },
    "rheumatologic_disease": {
        "snomed": RHEUM_SNOMED_IDS,
        "icd": RHEUM_ICD_IDS,
        "points": 1
    },
    "peptic_ulcer_disease": {
        "snomed": PUD_SNOMED_IDS,
        "icd": PUD_ICD_IDS,
        "points": 1
    },
    "diabetes": {
        "snomed": DM_SNOMED_IDS,
        "icd": DM_ICD_IDS,
        "points": 1
    },
    "renal_disease": {
        "snomed": RENAL_SNOMED_IDS,
        "icd": RENAL_ICD_IDS,
        "points": 2
    },
    "hemiplegia_paraplegia": {
        "snomed": HEMI_SNOMED_IDS,
        "icd": HEMI_ICD_IDS,
        "points": 2
    },
    "liver_disease": {
        "snomed": LIVER_SNOMED_IDS,
        "icd": LIVER_ICD_IDS,
        "points": 1   # forced to 1 as you requested
    },
    "malignancy": {
        "snomed": MALIG_SNOMED_IDS,
        "icd": MALIG_ICD_IDS,
        "points": "special"
    },
    "aids": {
        "snomed": AIDS_SNOMED_IDS,
        "icd": AIDS_ICD_IDS,
        "points": 6
    },
    "leukemia": {
        "snomed": LEUK_SNOMED_IDS,
        "icd": LEUK_ICD_IDS,
        "points": 2
    },
    "lymphoma": {
        "snomed": LYM_SNOMED_IDS,
        "icd": LYM_ICD_IDS,
        "points": 2
    },
    "dementia": {
        "snomed": DEM_SNOMED_IDS,
        "icd": DEM_ICD_IDS,
        "points": 1
    }
}

In [ ]:
def query_conditions(workspace, snomed_ids, icd_ids):
    sql = f"""
SELECT
    c.person_id,
    c.condition_concept_id,
    cc.concept_name AS standard_concept_name,
    cc.concept_code AS standard_concept_code,
    cc.vocabulary_id AS standard_vocabulary,
    c.condition_source_concept_id,
    cs.concept_name AS source_concept_name,
    cs.concept_code AS source_concept_code,
    cs.vocabulary_id AS source_vocabulary,
    c.condition_start_datetime,
    c.condition_end_datetime
FROM `{os.environ["WORKSPACE_CDR"]}.condition_occurrence` c
LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` cc
    ON c.condition_concept_id = cc.concept_id
LEFT JOIN `{os.environ["WORKSPACE_CDR"]}.concept` cs
    ON c.condition_source_concept_id = cs.concept_id
WHERE c.condition_concept_id IN UNNEST({snomed_ids})
        OR c.condition_source_concept_id IN UNNEST({icd_ids})
    """
    return pd.read_gbq(sql, dialect="standard")

def build_comorbidity_df(name, cfg, workspace):
    df = query_conditions(
        workspace,
        cfg["snomed"],
        cfg["icd"]
    )

    df["comorbidity"] = name

    # one row per person
    df = (
        df.sort_values("condition_start_datetime")
          .drop_duplicates("person_id")
          .reset_index(drop=True)
    )

    return df
comorbidity_dfs = {}

for name, cfg in COMORBIDITIES.items():
    comorbidity_dfs[name] = build_comorbidity_df(
        name,
        cfg,
        os.environ["WORKSPACE_CDR"]
    )

In [ ]:
# for name, cfg in COMORBIDITIES.items():
#     if name=='dementia':
#         comorbidity_dfs[name] = build_comorbidity_df(
#         name,
#         cfg,
#         os.environ["WORKSPACE_CDR"]
#     )

In [ ]:
import gzip
import pickle
# with gzip.open(f"charlston_comorbidity.pkl.gz", "wb") as f:
#     pickle.dump(comorbidity_dfs, f)
with gzip.open(f"charlston_comorbidity.pkl.gz", "rb") as f:
    comorbidity_dfs = pickle.load(f)


comorbidity_dfs["renal_disease"]

In [ ]:
import pandas as pd
import os
dataset_13847333_age_sql = """
    SELECT
        * 
    FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_ehr_data = 1 """

df_age = pd.read_gbq(
    dataset_13847333_age_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")
df_age.head(5)
df_age["age_at_consent_1"] = df_age["age_at_consent"]+1
df_age["age_at_consent_2"] = df_age["age_at_consent"]+2
def age_score(age):
    if age >= 80:
        return 4
    elif age >= 70:
        return 3
    elif age >= 60:
        return 2
    elif age >= 50:
        return 1
    else:
        return 0
age_df = (
    df_age[["person_id", "age_at_consent_2"]]
    .assign(age_score=lambda x: x["age_at_consent_2"].apply(age_score))
)

In [ ]:

# df_age['age_date'] = (
#     pd.to_datetime(df_age['dob']) +
#     pd.to_timedelta(df_age['age_at_consent'] * 365.25, unit='D')
# )
# df_age[['person_id','age_at_cdr','age_date']]


In [ ]:
#new line/updated
def filter_before_event(df, index_dates, date_col):
    df = df.merge(index_dates, on="person_id", how="left")

    df[date_col] = pd.to_datetime(df[date_col], utc=True).dt.tz_localize(None)
    df["date_consider"] = pd.to_datetime(df["date_consider"]).dt.tz_localize(None)

    return df[df[date_col] <= df["date_consider"]]


def score_standard(df, name, points, index_dates, date_col):
    df = filter_before_event(df, index_dates, date_col)

    return (
        df.groupby("person_id", as_index=False)
          .size()
          .assign(**{f"{name}_score": points})
          [["person_id", f"{name}_score"]]
    )

# def score_standard(df, name, points):
#     return (
#         df[["person_id"]]
#         .assign(**{f"{name}_score": points})
#     )
METASTATIC_KEYWORDS = ["metast", "secondary", "stage iv", "stage 4"]

# def score_malignancy(df):
#     df = df.copy()

#     df["is_metastatic"] = (
#         df["source_concept_name"]
#         .str.lower()
#         .str.contains("|".join(METASTATIC_KEYWORDS), na=False)
#     )

#     scored = (
#         df.groupby("person_id")["is_metastatic"]
#         .max()
#         .reset_index()
#     )

#     scored["malignancy_score"] = scored["is_metastatic"].map(
#         lambda x: 6 if x else 2
#     )

#     return scored[["person_id", "malignancy_score"]]

def score_malignancy(df, index_dates, date_col):
    df = filter_before_event(df, index_dates, date_col)

    df["is_metastatic"] = (
        df["source_concept_name"]
        .str.lower()
        .str.contains("|".join(METASTATIC_KEYWORDS), na=False)
    )

    scored = (
        df.groupby("person_id")["is_metastatic"]
          .max()
          .reset_index()
    )

    scored["malignancy_score"] = scored["is_metastatic"].map(
        lambda x: 6 if x else 2
    )

    return scored[["person_id", "malignancy_score"]]

# score_dfs = []

# for name, cfg in COMORBIDITIES.items():
#     df = comorbidity_dfs[name]

#     if cfg["points"] == "special":
#         score_dfs.append(score_malignancy(df))
#     else:
#         score_dfs.append(score_standard(df, name, cfg["points"]))

# from functools import reduce

# charlson_scores = reduce(
#     lambda l, r: l.merge(r, on="person_id", how="outer"),
#     score_dfs
# ).fillna(0)


#all of aou
score_dfs = []

for name, cfg in COMORBIDITIES.items():
    df = comorbidity_dfs[name]

    if cfg["points"] == "special":
        score_dfs.append(
            score_malignancy(df, cohort_meta_df, "condition_start_datetime")
        )
    else:
        score_dfs.append(
            score_standard(
                df, name, cfg["points"],
                cohort_meta_df, "condition_start_datetime"
            )
        )

from functools import reduce

charlson_scores = (
    reduce(
        lambda l, r: l.merge(r, on="person_id", how="outer"),
        score_dfs
    )
    .fillna(0)
)

#fitbit
score_dfs_f = []

for name, cfg in COMORBIDITIES.items():
    df = comorbidity_dfs[name]

    if cfg["points"] == "special":
        score_dfs_f.append(
            score_malignancy(df, cohort_fitbit_meta_df, "condition_start_datetime")
        )
    else:
        score_dfs_f.append(
            score_standard(
                df, name, cfg["points"],
                cohort_fitbit_meta_df, "condition_start_datetime"
            )
        )

from functools import reduce

charlson_scores_f = (
    reduce(
        lambda l, r: l.merge(r, on="person_id", how="outer"),
        score_dfs_f
    )
    .fillna(0)
)




# charlson_scores["charlson_score"] = (
#     charlson_scores.drop(columns="person_id").sum(axis=1)
# )

In [ ]:
#new line/updated
charlson = (
    charlson_scores
    .merge(age_df, on="person_id", how="right")
    .fillna(0)
)

charlson["charlson_score"] = (
    charlson
    .drop(columns=["person_id", "age_at_consent_2"])
    .sum(axis=1)
)
charlson = charlson.merge(cohort_meta_df,on = 'person_id')

In [ ]:
#new line/updated
charlson_f = (
    charlson_scores_f
    .merge(age_df, on="person_id", how="right")
    .fillna(0)
)

charlson_f["charlson_score"] = (
    charlson_f
    .drop(columns=["person_id", "age_at_consent_2"])
    .sum(axis=1)
)
charlson_f = charlson_f.merge(cohort_fitbit_meta_df,on = 'person_id')

In [ ]:
import matplotlib.pyplot as plt
plt.hist(charlson['charlson_score'])
plt.hist(charlson_f['charlson_score'])

In [ ]:
#new line/updated
plt.Figure()
bins = np.arange(
    charlson['charlson_score'].min(),
    charlson['charlson_score'].max() + 1
)
# plt.hist(charlson[charlson['meta_label']==2]['charlson_score'],bins=bins,\
#         label = 'incident',alpha = 0.4)
# plt.hist(charlson[charlson['meta_label']==1]['charlson_score'],bins=bins,\
#         label = 'prevalent',alpha = 0.4)
plt.hist(charlson[charlson['meta_label']==0]['charlson_score'],bins=bins,\
        label = 'controls',alpha = 0.4,color = 'green')
plt.xticks(
    np.arange(
        charlson['charlson_score'].min(),
        charlson['charlson_score'].max() + 1,
        2
    )
)
plt.legend()

In [ ]:
#new line/updated
plt.Figure()
bins = np.arange(
    charlson_f['charlson_score'].min(),
    charlson_f['charlson_score'].max() + 1
)
# plt.hist(charlson_f[charlson_f['meta_label']==2]['charlson_score'],bins=bins,\
#         label = 'incident',alpha = 0.4)
# plt.hist(charlson_f[charlson_f['meta_label']==1]['charlson_score'],bins=bins,\
#         label = 'prevalent',alpha = 0.4)
plt.hist(charlson_f[charlson_f['meta_label']==0]['charlson_score'],bins=bins,\
        label = 'controls',alpha = 0.4,color = 'green')

plt.xticks(
    np.arange(
        charlson_f['charlson_score'].min(),
        charlson_f['charlson_score'].max() + 1,
        2
    )
)
plt.legend()

In [ ]:
wear = pd.read_csv(f'gs://fc-secure-19850434-d631-4db8-b9e0-344c61477671/stratification_person_stats.tsv',sep='\t')
charlson_f = charlson_f.merge(wear[['wear_consent','person_id']],on = 'person_id')

In [ ]:
#new line/updated
plt.Figure()
bins = np.arange(
    charlson_f['charlson_score'].min(),
    charlson_f['charlson_score'].max() + 1
)
colors = {
    0: 'tab:green',   # controls
    1: 'tab:orange',     # prevalent
    2: 'tab:blue'     # incident
}

# for meta in [0, 1, 2]:
for meta in [0]:
    subset = charlson_f[charlson_f['meta_label'] == meta]
    
    
    
    plt.hist(
        subset[subset['wear_consent'] == 0]['charlson_score'],
        bins=bins,
        alpha=0.2,
        color=colors[meta],
        hatch='//' ,
        label=f"{'incident' if meta==2 else ('prevalent' if meta==1 else 'controls')} (BYOD)"
    )
    
    plt.hist(
        subset[subset['wear_consent'] == 1]['charlson_score'],
        bins=bins,
        alpha=0.4,
        color=colors[meta],
        label=f"{'incident' if meta==2 else ('prevalent' if meta==1 else 'controls')} (Wear study)"
    )
    
plt.xticks(
    np.arange(
        charlson_f['charlson_score'].min(),
        charlson_f['charlson_score'].max() + 1,
        2
    )
)
plt.legend()

In [ ]:
#new line/updated
charlson_f_n = charlson_f.merge(wear,on = ['person_id', 'wear_consent'])

In [ ]:
#new line/updated
def print_ots_table1(df_new, filter_name, filter_):
    
    df_c_new = df_new[df_new[filter_name].isin(filter_)].copy()
    
    print(f"No:\n \
    New: {len(set(df_c_new['person_id']))}")
    
    for i,j in zip([18,45],[45,66]):
        print(f"Age no {i},{j-1}:\n \
        New: {len(set(df_c_new[(df_c_new['age_at_cdr']>=i)&(df_c_new['age_at_cdr']<j)]['person_id']))}\
        ({np.round(len(set(df_c_new[(df_c_new['age_at_cdr']>=i)&(df_c_new['age_at_cdr']<j)]['person_id']))/len(df_c_new)*100,1)})%")
    k = 66
    print(f"Age no {k-1}:\n \
        New: {len(set(df_c_new[(df_c_new['age_at_cdr']>=k)]['person_id']))}\
        ({np.round(len(set(df_c_new[(df_c_new['age_at_cdr']>=k)]['person_id']))/len(df_c_new)*100,1)})%")
    
    for i in ['Female','Male']:
        print(f"Sex: {i}:\n \
        New: {len(set(df_c_new[(df_c_new['sex_at_birth']==i)]['person_id']))}\
        ({np.round(len(set(df_c_new[(df_c_new['sex_at_birth']==i)]['person_id']))/len(df_c_new)*100,1)})%")
    
    for i in ['White','Black or African American','Asian','Other']:
        print(f"Race: {i}:\n \
        New: {len(set(df_c_new[(df_c_new['Race']==i)]['person_id']))}\
        ({np.round(len(set(df_c_new[(df_c_new['Race']==i)]['person_id']))/len(df_c_new)*100,1)})%")
    
    for i in ['Hispanic or Latino','Not Hispanic or Latino']:
        print(f"Ethnicity: {i}:\n \
        New: {len(set(df_c_new[(df_c_new['ethnicity']==i)]['person_id']))}\
        ({np.round(len(set(df_c_new[(df_c_new['ethnicity']==i)]['person_id']))/len(df_c_new)*100,1)})%")
    
    print(f"Ethnicity: Other:\n \
        New: {len(set(df_c_new[~(df_c_new['ethnicity'].isin(['Hispanic or Latino','Not Hispanic or Latino']))]['person_id']))}\
        ({np.round(len(set(df_c_new[~(df_c_new['ethnicity'].isin(['Hispanic or Latino','Not Hispanic or Latino']))]['person_id']))/len(df_c_new)*100,1)})%")
    
    for i in ['College Degree or Advanced','Some college degree','No college degree','Undefined']:
        print(f"Education: {i}:\n \
        New: {len(set(df_c_new[(df_c_new['education_level']==i)]['person_id']))}\
        ({np.round(len(set(df_c_new[(df_c_new['education_level']==i)]['person_id']))/len(df_c_new)*100,1)})%")
    
    for i,j in zip(['100 Cigs Lifetime: No','100 Cigs Lifetime: Yes'],['Never','Ever (>=100 cigarettes)']):
        print(f"Smoking: {j}:\n \
        New: {len(set(df_c_new[(df_c_new['answer_s']==i)]['person_id']))}\
        ({np.round(len(set(df_c_new[(df_c_new['answer_s']==i)]['person_id']))/len(df_c_new)*100,1)})%")
    
    print(f"Smoking: Other:\n \
        New: {len(set(df_c_new[~(df_c_new['answer_s'].isin(['100 Cigs Lifetime: No','100 Cigs Lifetime: Yes']))]['person_id']))}\
        ({np.round(len(set(df_c_new[~(df_c_new['answer_s'].isin(['100 Cigs Lifetime: No','100 Cigs Lifetime: Yes']))]['person_id']))/len(df_c_new)*100,1)})%")
    
    for i,j in zip(['Drink Frequency Past Year: Never','Drink Frequency Past Year: Monthly Or Less',\
                   'Drink Frequency Past Year: 2 to 4 Per Month','Drink Frequency Past Year: 2 to 3 Per Week',\
                   'Drink Frequency Past Year: 4 or More Per Week'],\
                   ['Never','Monthly Or Less','2-4/Month','2-3/Week','>=4/Week']):
        print(f"Alcohol: {j}:\n \
        New: {len(set(df_c_new[(df_c_new['answer']==i)]['person_id']))}\
        ({np.round(len(set(df_c_new[(df_c_new['answer']==i)]['person_id']))/len(df_c_new)*100,1)})%")
    
    print(f"Alcohol: Other:\n \
        New: {len(set(df_c_new[(df_c_new['answer'].isin(['PMI: Dont Know','PMI: Skip','PMI: Prefer Not To Answer']))]['person_id']))}\
        ({np.round(len(set(df_c_new[(df_c_new['answer'].isin(['PMI: Dont Know','PMI: Skip','PMI: Prefer Not To Answer']))]['person_id']))/len(df_c_new)*100,1)})%")
    
#     for i in ['<= 25','25–30','> 30']:
#         print(f"BMI: {i}:\n \
#         New: {len(set(df_c_new[(df_c_new['BMI Category']==i)]['person_id']))}\
#         ({np.round(len(set(df_c_new[(df_c_new['BMI Category']==i)]['person_id']))/len(df_c_new[~df_c_new['BMI Category'].isna()])*100,1)})%")
    
    
    for i,j in zip(['htn','dm','hld','cad','hf','ckd'],\
                   ['Hypertension','Diabetes','Hyperlipidemia or Dyslipidemia',\
                    'Coronary Artery Disease','Heart Failure','Chronic Kidney Disease']):
        print(f"{j}:\n \
        New: {len(set(df_c_new[(df_c_new[i]==1)]['person_id']))}\
        ({np.round(len(set(df_c_new[(df_c_new[i]==1)]['person_id']))/len(df_c_new)*100,1)})%")
    
    
    
# print_ots_table1(charlson_f_n,'wear_consent', [0])  
# print_ots_table1(charlson_f_n,'wear_consent', [1])  
           

### BCS score check

In [ ]:
# dfs_all = pd.read_csv('bcs_021126.csv')
# cols = ['smoking', 'alcohol', 'pa_vigorous', 'pa_moderate',
#        'sleep', 'stress', 'social', 'meaning', 'hearing', 'blood_pressure',
#        'glucose', 'cholesterol']
# cols_n = ['smoking', 'alcohol', 'pa_vigorous', 'pa_moderate',
#        'sleep', 'stress', 'social', 'meaning', 'hearing']
# cols_v = ['blood_pressure',
#        'glucose', 'cholesterol']

# bad_vals = ['exclude', 'no close value to match']

# is_bad_all = dfs_all[cols].isin(bad_vals).all(axis=1)
# has_good = (~dfs_all[cols].isin(bad_vals)).any(axis=1)

# dfs_all_a = dfs_all[has_good]

# bcs_all = dfs_all_a.merge(cohort_meta_df,on = 'person_id')
# bcs_f = dfs_all_a.merge(cohort_fitbit_meta_df,on = 'person_id')


# bcs_p_all = dfs_all_a[has_good_all].merge(cohort_meta_df,on = 'person_id')
# bcs_p_f = dfs_all_a[has_good_all].merge(cohort_fitbit_meta_df,on = 'person_id')

dfs_all = pd.read_csv('bcs_031826.csv')
cols = ['smoking', 'alcohol', 'pa_vigorous', 'pa_moderate',
       'sleep', 'stress', 'social', 'meaning', 'hearing', 'blood_pressure',
       'glucose', 'cholesterol']
cols_n = ['smoking', 'alcohol', 'pa_vigorous', 'pa_moderate',
       'sleep', 'stress', 'social', 'meaning', 'hearing']
cols_v = ['blood_pressure',
       'glucose', 'cholesterol']

bad_vals = ['exclude']
bad_vals_s = ['exclude', 'no close value to match']

len(dfs_all[dfs_all[cols].isin(bad_vals_s).all(axis=1)]),\
len(dfs_all)- len(dfs_all[dfs_all[cols].isin(bad_vals_s).all(axis=1)]),\
len(dfs_all) - len(dfs_all[dfs_all[cols_v].isin(bad_vals_s).all(axis=1)]),\
len(dfs_all[~dfs_all[cols].isin(bad_vals_s).any(axis=1)]),\
len(dfs_all[dfs_all[cols].isin(bad_vals).all(axis=1)]),\
len(dfs_all)- len(dfs_all[dfs_all[cols].isin(bad_vals).all(axis=1)]),\
len(dfs_all) - len(dfs_all[dfs_all[cols_v].isin(bad_vals).all(axis=1)]),\
len(dfs_all[~dfs_all[cols].isin(bad_vals).any(axis=1)]),\




In [ ]:
import matplotlib.pyplot as plt
cols = [
    'smoking', 'alcohol', 'pa_vigorous', 'pa_moderate',
    'sleep', 'stress', 'social', 'meaning', 'hearing',
    'blood_pressure', 'glucose', 'cholesterol'
]

summary = []

for c in cols:
    vals = dfs_all[c]
    
    numeric = pd.to_numeric(vals, errors='coerce')
    
    n_numeric = numeric.notna().sum() + vals.isin(['no close value to match']).sum()
    n_exclude = vals.isin(['exclude']).sum()
    n_custom = numeric.notna().sum()
    n_total = len(vals)
    
    summary.append([c, n_numeric, n_exclude, n_total, n_custom])
summary_df = pd.DataFrame(summary, columns=[
    'metric', 'n_numeric', 'n_exclude', 'n_total', 'n_custom'
])

summary_df["p_numeric"] = summary_df["n_numeric"] / summary_df["n_total"]
summary_df["p_exclude"] = summary_df["n_exclude"] / summary_df["n_total"]
summary_df["p_custom"] = summary_df["n_custom"] / summary_df["n_total"]

summary_df = summary_df.sort_values("p_numeric", ascending=False).reset_index(drop=True)
x = np.arange(len(summary_df))
width = 0.35

plt.figure(figsize=(13,5))

bars1 = plt.bar(x - width/2, summary_df["p_numeric"], width, label="Score available")
bars2 = plt.bar(x + width/2, summary_df["p_exclude"], width, label="Survey answered \n(unknown/ exclude)")

# ---- Annotate counts ----
for i, bar in enumerate(bars1):
    h = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        h,
        f"{summary_df['n_numeric'][i]:,}",
        ha='center',
        va='bottom',
        fontsize=8
    )

for i, bar in enumerate(bars2):
    h = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        h,
        f"{summary_df['n_exclude'][i]:,}",
        ha='center',
        va='bottom',
        fontsize=8
    )

plt.xticks(x, summary_df["metric"], rotation=45)
plt.ylabel("Proportion")
plt.xlabel("Survey")
# plt.title("Proportion of Numeric vs Excluded Scores")
plt.legend()
plt.grid(axis='y')
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout()
plt.show()


# ---- filter only needed metrics ----
subset_metrics = ['blood_pressure', 'glucose', 'cholesterol']
df_plot = summary_df[summary_df['metric'].isin(subset_metrics)].copy()

# ---- positions ----
x = np.arange(len(df_plot))
width = 0.25

plt.figure(figsize=(8,5))

bars1 = plt.bar(x - width, df_plot["p_numeric"], width, label="Score available")
bars2 = plt.bar(x, df_plot["p_exclude"], width, label="Survey answered\n(exclude)")
bars3 = plt.bar(x + width, df_plot["p_custom"], width, label="Correlated with \nmeasurements")

# ---- annotate counts for first two ----
for i, bar in enumerate(bars1):
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, h,
             f"{df_plot['n_numeric'].iloc[i]:,}",
             ha='center', va='bottom', fontsize=8)

for i, bar in enumerate(bars2):
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, h,
             f"{df_plot['n_exclude'].iloc[i]:,}",
             ha='center', va='bottom', fontsize=8)

# ---- annotate custom (optional) ----
for i, bar in enumerate(bars3):
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, h,
             f"{df_plot['n_custom'].iloc[i]:,}",
             ha='center', va='bottom', fontsize=8)

# ---- formatting ----
plt.xticks(x, df_plot["metric"], rotation=45)
plt.ylabel("Proportion")
plt.xlabel("Survey")

plt.grid(axis='y')
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout()
plt.show()


In [ ]:
cols = ['smoking', 'alcohol', 'pa_vigorous', 'pa_moderate',
       'sleep', 'stress', 'social', 'meaning', 'hearing', 'blood_pressure',
       'glucose', 'cholesterol']
cols_n = ['smoking', 'alcohol', 'pa_vigorous', 'pa_moderate',
       'sleep', 'stress', 'social', 'meaning', 'hearing']
cols_v = ['blood_pressure',
       'glucose', 'cholesterol']

# bad_vals = ['exclude', 'no close value to match']
bad_vals = ['exclude']

is_bad_all = dfs_all[cols].isin(bad_vals).all(axis=1)
has_good = (~dfs_all[cols].isin(bad_vals)).any(axis=1)
has_good_all = (~dfs_all[cols].isin(bad_vals)).all(axis=1)


bcs_all = dfs_all[has_good].merge(cohort_meta_df,on = 'person_id')
bcs_f = dfs_all[has_good].merge(cohort_fitbit_meta_df,on = 'person_id')


bcs_p_all = dfs_all[has_good_all].merge(cohort_meta_df,on = 'person_id')
bcs_p_f = dfs_all[has_good_all].merge(cohort_fitbit_meta_df,on = 'person_id')

In [ ]:
#new line/updated
plt.Figure()
bins = np.arange(
    bcs_f['total_score'].min(),
    bcs_f['total_score'].max() + 1, 2
)
plt.hist(bcs_f[bcs_f['meta_label']==2]['total_score'],bins=bins,\
        label = 'incident',alpha = 0.4)
plt.hist(bcs_f[bcs_f['meta_label']==1]['total_score'],bins=bins,\
        label = 'prevalent',alpha = 0.4)
# plt.hist(bcs_f[bcs_f['meta_label']==0]['total_score'],bins=bins,\
#         label = 'controls',alpha = 0.4,color = 'green')

plt.xticks(
    np.arange(
        bcs_f['total_score'].min(),
        bcs_f['total_score'].max() + 1,
        4
    )
)
plt.legend()

In [ ]:
#new line/updated
plt.Figure()
bins = np.arange(
    bcs_f['total_score'].min(),
    bcs_f['total_score'].max() + 1, 2
)
plt.hist(bcs_p_f[bcs_p_f['meta_label']==2]['total_score'],bins=bins,\
        label = 'incident',alpha = 0.4)
plt.hist(bcs_p_f[bcs_p_f['meta_label']==1]['total_score'],bins=bins,\
        label = 'prevalent',alpha = 0.4)
plt.hist(bcs_p_f[bcs_p_f['meta_label']==0]['total_score'],bins=bins,\
        label = 'controls',alpha = 0.4,color = 'green')

plt.xticks(
    np.arange(
        bcs_f['total_score'].min(),
        bcs_f['total_score'].max() + 1,
        4
    )
)
plt.legend()

In [ ]:
#new line/updated
plt.Figure()
bins = np.arange(
    bcs_all['total_score'].min(),
    bcs_all['total_score'].max() + 1, 2
)
plt.hist(bcs_all[bcs_all['meta_label']==2]['total_score'],bins=bins,\
        label = 'incident',alpha = 0.4)
plt.hist(bcs_all[bcs_all['meta_label']==1]['total_score'],bins=bins,\
        label = 'prevalent',alpha = 0.4)
plt.hist(bcs_all[bcs_all['meta_label']==0]['total_score'],bins=bins,\
        label = 'controls',alpha = 0.4,color = 'green')

plt.xticks(
    np.arange(
        bcs_all['total_score'].min(),
        bcs_all['total_score'].max() + 1,
        4
    )
)
plt.legend()

In [ ]:
#new line/updated
plt.Figure()
bins = np.arange(
    bcs_all['total_score'].min(),
    bcs_all['total_score'].max() + 1, 2
)
plt.hist(bcs_p_all[bcs_p_all['meta_label']==2]['total_score'],bins=bins,\
        label = 'incident',alpha = 0.4)
plt.hist(bcs_p_all[bcs_p_all['meta_label']==1]['total_score'],bins=bins,\
        label = 'prevalent',alpha = 0.4)
plt.hist(bcs_p_all[bcs_p_all['meta_label']==0]['total_score'],bins=bins,\
        label = 'controls',alpha = 0.4,color = 'green')

plt.xticks(
    np.arange(
        bcs_all['total_score'].min(),
        bcs_all['total_score'].max() + 1,
        4
    )
)
plt.legend()